# Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Configuration](#2-configuration)
3. [Reproducibility and Device Setup](#3-reproducibility-and-device-setup)
4. [Dataset Discovery and Metadata Cache](#4-dataset-discovery-and-metadata-cache)
5. [Dependencies and Imports](#5-dependencies-and-imports)
6. [Data Loading and Preprocessing](#6-data-loading-and-preprocessing)
7. [Model Architecture](#7-model-architecture)
8. [Experiment Tracking](#8-experiment-tracking)
9. [Training Utilities](#9-training-utilities)
10. [Training Loop](#10-training-loop)
11. [Evaluation](#11-evaluation) (includes threshold sweep, pixel AUC, confusion matrix, forgery-type, mask-size, shortcut checks)
12. [Visualization of Predictions](#12-visualization-of-predictions) (includes ELA visualization)
13. [Advanced Analysis](#13-advanced-analysis) (failure cases)
14. [Explainability](#14-explainability) (Grad-CAM)
15. [Robustness Testing](#15-robustness-testing)
16. [Inference Examples](#16-inference-examples)
17. [Model Card and Experiment Report](#17-model-card-and-experiment-report)
18. [Conclusion](#18-conclusion)

# Tampered Image Detection and Localization (vK.11.1)

This Kaggle-first notebook presents a complete assignment submission for tampered image
detection and tampered region localization.

**Architecture upgrades in vK.11.1** (from audit findings + Docs v11)
- **Pretrained ResNet34 encoder** via `segmentation_models_pytorch` (SMP) — proven in v6.5 to achieve Tam-F1=0.41 vs 0.22 from scratch
- **ELA (Error Level Analysis)** as 4th input channel — amplifies JPEG compression inconsistencies between authentic and tampered regions
- **Edge supervision loss** (Sobel-based boundary BCE) — improves boundary delineation around tampered regions
- **AdamW with differential learning rates** — encoder=1e-4, decoder/heads=1e-3 (from v6.5 pattern)
- **ReduceLROnPlateau scheduler** — replaces CosineAnnealing double-cycle bug from vK.10.6
- **Gradient accumulation** — effective batch = batch_size × accumulation_steps
- **Encoder freeze** for first 2 epochs — protects pretrained BatchNorm statistics
- **Per-sample Dice loss** — fixes batch-level bias toward large masks (from v8)

**Evaluation suite carried forward from vK.10.6** (12 features)
- Threshold sweep, pixel-level AUC, confusion matrix, ROC curve, PR curve
- Forgery-type breakdown, mask-size stratification, shortcut learning checks
- Grad-CAM explainability, robustness testing (8 conditions), failure case analysis
- Data leakage verification (path overlap assertion)

**Engineering carried forward from vK.10.5/vK.10.6**
- Multi-GPU training with `nn.DataParallel`
- Centralized CONFIG dictionary for all hyperparameters
- Full reproducibility seeding (Python, NumPy, PyTorch CPU/CUDA)
- Automatic Mixed Precision (AMP) for faster training
- Three-file checkpoint system with automatic resume
- Early stopping based on tampered-only Dice coefficient
- Metadata caching and VRAM-based batch size auto-adjustment

**Notebook deliverables**
- Image-level tamper detection through the classifier head
- Pixel-level tampered region localization through the segmentation branch
- ELA forensic signal visualization
- Comprehensive 12-point evaluation suite with robustness testing

## Project Objectives: Fulfilled vs Remaining

| Requirement | Status | Evidence |
|---|---|---|
| Dataset: authentic + tampered + masks | Fulfilled | CASIA dataset with IMAGE/MASK dirs |
| Model performs detection + localization | Fulfilled | TamperDetector dual-head (SMP UNet + classifier) |
| Pretrained encoder for transfer learning | Fulfilled | ResNet34 (ImageNet) via SMP |
| ELA forensic preprocessing | Fulfilled | 4th input channel from JPEG re-save analysis |
| Evaluation with Dice / IoU / F1 | Fulfilled | Tampered-only and all-sample metrics |
| Visual results (Original, GT, Pred, Overlay) | Fulfilled | Submission prediction grid |
| Single notebook | Fulfilled | All code in one notebook |
| Reproducibility | Fulfilled | Full seeding + checkpoint resume |
| AMP training | Fulfilled | autocast + GradScaler |
| Early stopping | Fulfilled | Patience-based on tampered Dice |
| Multi-GPU utilization | Fulfilled | nn.DataParallel across T4 x2 |
| Threshold optimization | Fulfilled | 50-point sweep on validation set |
| Robustness testing | Fulfilled | 8 degradation conditions with deltas |
| Grad-CAM explainability | Fulfilled | Encoder layer4 heatmaps |
| Confusion matrix + ROC/PR | Fulfilled | seaborn heatmap + sklearn curves |
| Forgery-type breakdown | Fulfilled | Splicing vs copy-move metrics |
| Data leakage verification | Fulfilled | Path overlap assertion |
| Shortcut learning checks | Fulfilled | Mask randomization + boundary sensitivity |
| Failure case analysis | Fulfilled | 10 worst predictions annotated |
| Pixel-level AUC-ROC | Fulfilled | Threshold-independent localization metric |
| Edge supervision loss | Fulfilled | Sobel-based boundary BCE |
| Gradient accumulation | Fulfilled | Effective batch = batch_size x accumulation_steps |
| Encoder freeze warmup | Fulfilled | First 2 epochs frozen |

## 1. Environment Setup

In [1]:
import subprocess
import sys
import os
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
KAGGLE_DATASET_SLUG = "harshv777/casia2-0-upgraded-dataset"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")
ATTACHED_DATASET_DIR = KAGGLE_INPUT_DIR / "casia2-0-upgraded-dataset"
DRIVE_SEARCH_ROOTS = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]

# Install segmentation-models-pytorch (not pre-installed on Kaggle or Colab)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "segmentation-models-pytorch",
])

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "albumentations==1.3.1", "opencv-python-headless==4.10.0.84", "kaggle",
    ])

KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("KAGGLE_INPUT_DIR:", KAGGLE_INPUT_DIR)
print("KAGGLE_WORKING_DIR:", KAGGLE_WORKING_DIR)
print("ATTACHED_DATASET_DIR:", ATTACHED_DATASET_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.3 MB/s eta 0:00:00
IN_COLAB: False
KAGGLE_INPUT_DIR: /kaggle/input
KAGGLE_WORKING_DIR: /kaggle/working
ATTACHED_DATASET_DIR: /kaggle/input/casia2-0-upgraded-dataset


In [2]:
def has_valid_layout(path):
    """Check for IMAGE/ and MASK/ subdirectories."""
    if not path or not path.is_dir():
        return False
    children = {c.name.lower() for c in path.iterdir() if c.is_dir()}
    return "image" in children and "mask" in children


def find_in_kaggle_input(input_dir):
    """Search /kaggle/input/ for any attached dataset with IMAGE+MASK layout."""
    if input_dir is None or not input_dir.exists() or not input_dir.is_dir():
        return None
    preferred, fallback = [], []
    # Exact slug match
    exact = input_dir / "casia2-0-upgraded-dataset"
    if exact.exists() and has_valid_layout(exact):
        preferred.append(exact)
    # Direct children
    for child in sorted(input_dir.iterdir()):
        if child.is_dir() and has_valid_layout(child):
            fallback.append(child)
    # Recursive scan (Kaggle may nest under /kaggle/input/datasets/user/slug/)
    for d in sorted(input_dir.rglob("*")):
        if d.is_dir() and has_valid_layout(d):
            if "casia" in d.name.lower():
                preferred.append(d)
            else:
                fallback.append(d)
    candidates = preferred or fallback
    return candidates[0] if candidates else None


def find_in_drive(roots):
    """Search Google Drive mount points for the dataset folder."""
    for root in roots:
        if not root.exists():
            continue
        candidate = root / "casia2-0-upgraded-dataset"
        if candidate.exists() and has_valid_layout(candidate):
            return candidate
        for match in root.rglob("*casia*"):
            if match.is_dir() and has_valid_layout(match):
                return match
    return None


def download_from_kaggle(slug, target_dir):
    """Download dataset via Kaggle API to /kaggle/working/ (writable)."""
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        # Download to writable directory, NOT /kaggle/input/ (read-only on Kaggle)
        download_dir = KAGGLE_WORKING_DIR / "_dataset_download"
        download_dir.mkdir(parents=True, exist_ok=True)
        api.dataset_download_files(slug, path=str(download_dir), unzip=True, quiet=False)
        if has_valid_layout(download_dir):
            return download_dir
        for d in download_dir.rglob("*"):
            if d.is_dir() and has_valid_layout(d):
                return d
    except Exception as e:
        print(f"Kaggle API fallback failed: {e}")
    return None


In [3]:
# DEBUG: show what Kaggle attached under /kaggle/input/
if KAGGLE_INPUT_DIR.exists():
    print("Contents of", KAGGLE_INPUT_DIR)
    for p in sorted(KAGGLE_INPUT_DIR.iterdir()):
        print(f"  {p.name}/  ->  {[c.name for c in p.iterdir() if c.is_dir()][:10] if p.is_dir() else 'file'}")
else:
    print(f"{KAGGLE_INPUT_DIR} does not exist")

dataset_dir = None

# 1) Kaggle attached dataset (default) — scan all attached datasets
dataset_dir = find_in_kaggle_input(KAGGLE_INPUT_DIR)
if dataset_dir:
    print(f"Using attached Kaggle dataset: {dataset_dir}")

# 2) Google Drive fallback (Colab only)
if dataset_dir is None and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        found = find_in_drive(DRIVE_SEARCH_ROOTS)
        if found:
            dataset_dir = found
            print(f"Using Google Drive dataset: {dataset_dir}")
    except Exception as e:
        print(f"Google Drive fallback skipped: {e}")

# 3) Kaggle API download (last resort — writes to /kaggle/working/)
if dataset_dir is None:
    dataset_dir = download_from_kaggle(KAGGLE_DATASET_SLUG, KAGGLE_WORKING_DIR)
    if dataset_dir:
        print(f"Downloaded dataset via Kaggle API: {dataset_dir}")

if not dataset_dir or not has_valid_layout(dataset_dir):
    raise FileNotFoundError(
        "Dataset not found. Attach harshv777/casia2-0-upgraded-dataset on Kaggle, "
        "or place it in Google Drive, or configure Kaggle API credentials."
    )

print(f"Dataset root: {dataset_dir}")
print(f"  IMAGE dir: {(dataset_dir / 'IMAGE').exists()}")
print(f"  MASK dir:  {(dataset_dir / 'MASK').exists()}")


Contents of /kaggle/input
  datasets/  ->  ['harshv777']
Using attached Kaggle dataset: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset
Dataset root: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset
  IMAGE dir: True
  MASK dir:  True


## 2. Configuration

Centralized configuration dictionary. All hyperparameters are defined here
and referenced by name throughout the notebook. Documentation tables below
are generated from the same values to prevent mismatches.

In [4]:
import os
from pathlib import Path

SEED = 42

CONFIG = {
    # -- Data --
    'image_size': 256,
    'batch_size': 8,             # auto-adjusted based on GPU VRAM
    'num_workers': 4,
    'train_ratio': 0.70,

    # -- Model (SMP pretrained — from v6.5 + Docs v11) --
    'architecture': 'TamperDetector',
    'encoder_name': 'resnet34',
    'encoder_weights': 'imagenet',
    'in_channels': 4,            # RGB + ELA
    'n_classes': 1,              # segmentation output channels
    'n_labels': 2,               # classification labels (authentic/tampered)
    'dropout': 0.5,

    # -- Optimizer (AdamW with differential LR — from v6.5) --
    'optimizer': 'AdamW',
    'encoder_lr': 1e-4,          # pretrained encoder: lower LR
    'decoder_lr': 1e-3,          # decoder + heads: higher LR
    'weight_decay': 1e-4,
    'max_grad_norm': 5.0,

    # -- Scheduler (ReduceLROnPlateau — from v8) --
    'scheduler': 'ReduceLROnPlateau',
    'scheduler_patience': 3,
    'scheduler_factor': 0.5,

    # -- Loss --
    'alpha': 1.5,                # classification loss weight
    'beta': 1.0,                 # segmentation loss weight
    'gamma': 0.3,                # edge loss weight (NEW)
    'focal_gamma': 2.0,
    'seg_bce_weight': 0.5,
    'seg_dice_weight': 0.5,

    # -- Training --
    'max_epochs': 50,
    'patience': 10,              # early stopping patience
    'checkpoint_every': 10,      # periodic checkpoint interval
    'accumulation_steps': 4,     # gradient accumulation steps
    'encoder_freeze_epochs': 2,  # freeze encoder for first N epochs

    # -- Feature Flags --
    'use_amp': True,
    'use_wandb': True,
    'seg_threshold': 0.5,

    # -- ELA --
    'ela_quality': 90,           # JPEG re-save quality for ELA computation

    # -- Reproducibility --
    'seed': SEED,
}

# Determine Kaggle vs local paths
if os.path.exists('/kaggle/working'):
    KAGGLE_WORKING_DIR = Path('/kaggle/working')
    KAGGLE_INPUT_DIR = Path('/kaggle/input')
elif os.path.exists('/content/drive'):
    KAGGLE_WORKING_DIR = Path('/content/drive/MyDrive/BigVision')
    KAGGLE_INPUT_DIR = KAGGLE_WORKING_DIR / 'input'
else:
    KAGGLE_WORKING_DIR = Path('.')
    KAGGLE_INPUT_DIR = Path('./input')

CHECKPOINT_DIR = Path(KAGGLE_WORKING_DIR) / 'checkpoints'
RESULTS_DIR    = Path(KAGGLE_WORKING_DIR) / 'results'
PLOTS_DIR      = Path(KAGGLE_WORKING_DIR) / 'plots'

for d in [CHECKPOINT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Working directory: {KAGGLE_WORKING_DIR}')
print(f'Input directory:   {KAGGLE_INPUT_DIR}')

Working directory: /kaggle/working
Input directory:   /kaggle/input


### 2.1 Hyperparameter Summary

| Hyperparameter | Value | Source |
|---|---|---|
| `architecture` | TamperDetector (SMP UNet + classifier head) | NEW in vK.11.0 |
| `encoder_name` | resnet34 | From v6.5 (proven Tam-F1=0.41) |
| `encoder_weights` | imagenet | Pretrained transfer learning |
| `in_channels` | 4 (RGB + ELA) | NEW in vK.11.0 |
| `image_size` | 256 | Preserved from vK.10.6 |
| `batch_size` | 8 (auto-adjusted) | VRAM-based |
| `accumulation_steps` | 4 (effective batch = batch_size x 4) | NEW in vK.11.0 |
| `optimizer` | AdamW | From v6.5 |
| `encoder_lr` | 1e-4 | Differential LR (from v6.5) |
| `decoder_lr` | 1e-3 | Differential LR (from v6.5) |
| `scheduler` | ReduceLROnPlateau(patience=3, factor=0.5) | From v8 |
| `max_epochs` | 50 | Preserved |
| `patience` | 10 | Early stopping on tampered Dice |
| `encoder_freeze_epochs` | 2 | NEW in vK.11.0 |
| `alpha` (cls weight) | 1.5 | Preserved |
| `beta` (seg weight) | 1.0 | Preserved |
| `gamma` (edge weight) | 0.3 | NEW in vK.11.0 |
| `focal_gamma` | 2.0 | Preserved |
| `use_amp` | True | Preserved |
| `ela_quality` | 90 | NEW in vK.11.0 |

## 3. Reproducibility and Device Setup

Full reproducibility is enforced through seeded random number generators across
Python, NumPy, and PyTorch. GPU diagnostics confirm hardware capabilities and
auto-adjust the batch size based on available VRAM.

In [5]:
import random
import numpy as np
import torch


def set_seed(seed):
    """Set seeds for full reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()

    for gpu_id in range(n_gpus):
        gpu_name = torch.cuda.get_device_name(gpu_id)
        vram_gb = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
        print(f'GPU {gpu_id}:          {gpu_name} ({vram_gb:.1f} GB)')

    total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)) / 1e9

    # TF32 for faster matmul on Ampere+ (does not affect determinism)
    if hasattr(torch.backends.cuda.matmul, 'fp32_precision'):
        torch.backends.cuda.matmul.fp32_precision = 'tf32'
        torch.backends.cudnn.conv.fp32_precision = 'tf32'
    else:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    print(f'GPUs available: {n_gpus}')
    print(f'Total VRAM:     {total_vram_gb:.1f} GB')
    print(f'cuDNN:          {torch.backends.cudnn.enabled}')
    print(f'Deterministic:  {torch.backends.cudnn.deterministic}')
    print(f'AMP:            {"enabled" if CONFIG["use_amp"] else "disabled"}')

    # Auto-adjust batch size based on total VRAM (SMP model ~1.1 GB base)
    if total_vram_gb >= 28:
        CONFIG['batch_size'] = 32
    elif total_vram_gb >= 20:
        CONFIG['batch_size'] = 24
    elif total_vram_gb >= 15:
        CONFIG['batch_size'] = 16
    # else keep CONFIG default (8)
    print(f'Batch size (auto-adjusted for {n_gpus} GPU{"s" if n_gpus > 1 else ""}): {CONFIG["batch_size"]}')
    print(f'Effective batch size: {CONFIG["batch_size"] * CONFIG["accumulation_steps"]}')
else:
    n_gpus = 0
    print('WARNING: No GPU detected. Training will be extremely slow.')

print(f'Device: {device}')

GPU 0:          Tesla T4 (15.6 GB)
GPU 1:          Tesla T4 (15.6 GB)
GPUs available: 2
Total VRAM:     31.3 GB
cuDNN:          True
Deterministic:  True
AMP:            enabled
Batch size (auto-adjusted for 2 GPUs): 32
Effective batch size: 128
Device: cuda


The batch size is auto-adjusted based on total available GPU VRAM:
- **>=28 GB** (2x T4): `batch_size=32` (effective=128 with accumulation)
- **>=20 GB**: `batch_size=24` (effective=96)
- **>=15 GB** (single T4): `batch_size=16` (effective=64)
- **<15 GB**: `batch_size=8` (effective=32, default)

The effective hyperparameters logged to W&B reflect the adjusted value.

## 4. Dataset Discovery and Metadata Cache

The notebook operates on an image tampering dataset organized into authentic and
tampered images with corresponding binary ground truth masks.

**Metadata caching:** If a valid `image_mask_metadata.csv` already exists with the
expected row count, the scanning step is skipped entirely to reduce startup time.


#### 2.1.1 Locate IMAGE and MASK Directories

Scans the normalized Kaggle-style input path and searches recursively for the `IMAGE` and `MASK` folders.

In [6]:
import os
from pathlib import Path
import pandas as pd

# =========================
# 1) Discover the dataset root
# =========================

# Inspect the Kaggle input directory so the notebook can confirm the expected dataset is available.
INPUT_ROOT = KAGGLE_INPUT_DIR

print(f"Available datasets under {INPUT_ROOT}:")
for p in INPUT_ROOT.iterdir():
    if p.is_dir():
        print(" -", p.name)

# Use the Kaggle-first attached dataset path prepared in the environment setup cells.
DATASET_DIR = dataset_dir  # resolved in Environment Setup (Kaggle -> Drive -> API)

# Search recursively for IMAGE and MASK so the metadata build still works if the dataset is nested one level deeper.
IMAGE_DIR = None
MASK_DIR = None

for p in DATASET_DIR.rglob("*"):
    if p.is_dir() and p.name.lower() == "image":
        IMAGE_DIR = p
    if p.is_dir() and p.name.lower() == "mask":
        MASK_DIR = p

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)

assert IMAGE_DIR is not None, "Could not find the IMAGE directory. Verify the dataset folder name and structure."
assert MASK_DIR is not None, "Could not find the MASK directory. Verify the dataset folder name and structure."

Available datasets under /kaggle/input:
 - datasets
IMAGE_DIR: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE
MASK_DIR: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/MASK


#### 2.1.2 Build Metadata Table from Image-Mask Pairs

Iterates over authentic (`Au`) and tampered (`Tp`) subdirectories,
pairing each image with its corresponding mask and recording the image-level label.


In [7]:
# =========================
# Build metadata with caching
# =========================

output_csv = KAGGLE_WORKING_DIR / "image_mask_metadata.csv"

# Count images in dataset to validate cache
_au_count = sum(1 for f in (IMAGE_DIR / 'Au').iterdir() if f.is_file())
_tp_count = sum(1 for f in (IMAGE_DIR / 'Tp').iterdir() if f.is_file())
_expected_count = _au_count + _tp_count

df = None
if output_csv.exists():
    cached_df = pd.read_csv(output_csv)
    if len(cached_df) == _expected_count:
        print(f'Using cached metadata ({len(cached_df)} rows): {output_csv}')
        df = cached_df
    else:
        print(f'Cache stale ({len(cached_df)} rows vs {_expected_count} expected), rebuilding...')

if df is None:
    label_folders = {
        "Au": {"class_name": "authentic", "label": 0},
        "Tp": {"class_name": "tampered",  "label": 1},
    }
    valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
    rows = []

    for sub_name, info in label_folders.items():
        img_subdir = IMAGE_DIR / sub_name
        mask_subdir = MASK_DIR / sub_name
        if not img_subdir.exists():
            print(f'Warning: image folder does not exist: {img_subdir}')
            continue
        print(f'Scanning images in: {img_subdir}')
        for img_path in img_subdir.iterdir():
            if not img_path.is_file():
                continue
            if img_path.suffix.lower() not in valid_exts:
                continue
            mask_path = mask_subdir / img_path.name
            mask_exists = mask_path.exists()
            rows.append({
                "image_path": str(img_path),
                "mask_path": str(mask_path) if mask_exists else "",
                "mask_exists": int(mask_exists),
                "class_folder": sub_name,
                "class_name": info["class_name"],
                "label": info["label"],
            })

    print(f'Total images found: {len(rows)}')
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'CSV saved to: {output_csv}')

print(df.head())
print('\nCounts per class_name:')
print(df['class_name'].value_counts())
print('\nMissing masks:')
print(df[df['mask_exists'] == 0].head())

Scanning images in: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE/Au
Scanning images in: /kaggle/input/datasets/harshv777/casia2-0-upgraded-dataset/IMAGE/Tp
Total images found: 12614
CSV saved to: /kaggle/working/image_mask_metadata.csv
                                          image_path  \
0  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
1  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
2  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
3  /kaggle/input/datasets/harshv777/casia2-0-upgr...   
4  /kaggle/input/datasets/harshv777/casia2-0-upgr...   

                                           mask_path  mask_exists  \
0  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
1  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
2  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
3  /kaggle/input/datasets/harshv777/casia2-0-upgr...            1   
4  /kaggle/input/datasets/harshv777/casia2-0-upgr...          

### 2.2 Train / Validation / Test Split

Loads the metadata CSV, applies a stratified 70/15/15 split to preserve class balance
across train, validation, and test subsets, and saves each split to a separate CSV file.


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Attempt to load cached split CSVs
train_csv = os.path.join(KAGGLE_WORKING_DIR, 'train_metadata.csv')
val_csv   = os.path.join(KAGGLE_WORKING_DIR, 'val_metadata.csv')
test_csv  = os.path.join(KAGGLE_WORKING_DIR, 'test_metadata.csv')

if os.path.exists(train_csv) and os.path.exists(val_csv) and os.path.exists(test_csv):
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv)
    test_df  = pd.read_csv(test_csv)
    total_loaded = len(train_df) + len(val_df) + len(test_df)
    if total_loaded == len(df):
        print(f'Loaded cached splits: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    else:
        print(f'Cache stale ({total_loaded} vs {len(df)}), re-splitting...')
        train_df, temp_df = train_test_split(
            df, test_size=1 - CONFIG['train_ratio'], stratify=df['label'],
            random_state=CONFIG['seed'],
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=0.50, stratify=temp_df['label'],
            random_state=CONFIG['seed'],
        )
        train_df.to_csv(train_csv, index=False)
        val_df.to_csv(val_csv, index=False)
        test_df.to_csv(test_csv, index=False)
else:
    train_df, temp_df = train_test_split(
        df, test_size=1 - CONFIG['train_ratio'], stratify=df['label'],
        random_state=CONFIG['seed'],
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['label'],
        random_state=CONFIG['seed'],
    )
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)
    test_df.to_csv(test_csv, index=False)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 8829 | Val: 1892 | Test: 1893


### 4.4 Dataset Summary

Quick summary of dataset splits and class balance.

In [9]:
print(f"{'Split':<8} {'Total':>6} {'Authentic':>10} {'Tampered':>9}")
print('-' * 38)
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_auth = (split_df['label'] == 0).sum()
    n_tamp = (split_df['label'] == 1).sum()
    print(f'{name:<8} {len(split_df):>6} {n_auth:>10} {n_tamp:>9}')

Split     Total  Authentic  Tampered
--------------------------------------
Train      8829       5243      3586
Val        1892       1124       768
Test       1893       1124       769


### 4.5 Data Leakage Verification

Verify zero overlap between train/val/test splits at the path level. This catches silent bugs where images appear in multiple splits.

In [10]:
# ================== Data Leakage Verification ==================
train_paths = set(train_df['image_path'].tolist())
val_paths   = set(val_df['image_path'].tolist())
test_paths  = set(test_df['image_path'].tolist())

tv_overlap = train_paths & val_paths
tt_overlap = train_paths & test_paths
vt_overlap = val_paths & test_paths

assert len(tv_overlap) == 0, f"LEAK: {len(tv_overlap)} images in both train and val!"
assert len(tt_overlap) == 0, f"LEAK: {len(tt_overlap)} images in both train and test!"
assert len(vt_overlap) == 0, f"LEAK: {len(vt_overlap)} images in both val and test!"

total_unique = len(train_paths | val_paths | test_paths)
total_rows = len(train_df) + len(val_df) + len(test_df)
assert total_unique == total_rows, f"LEAK: {total_rows} rows but only {total_unique} unique paths!"

print("DATA LEAKAGE CHECK: PASS")
print(f"  Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
print(f"  Total unique paths: {total_unique} == Total rows: {total_rows}")

DATA LEAKAGE CHECK: PASS
  Train: 8829 | Val: 1892 | Test: 1893
  Total unique paths: 12614 == Total rows: 12614


## 5. Dependencies and Imports

All training, evaluation, and visualization imports are consolidated here
to avoid redundant imports scattered across multiple cells.

In [11]:
import cv2
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt

print('Imports complete.')
print(f'PyTorch: {torch.__version__}')
print(f'SMP: {smp.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

Imports complete.
PyTorch: 2.9.0+cu126
SMP: 0.5.0
CUDA available: True


## 6. Data Loading and Preprocessing

The source notebook loads metadata CSV files, builds PyTorch datasets with
Albumentations augmentation, and creates dataloaders for training, validation,
and test splits.


### 6.1 Image and Mask Transforms

Defines Albumentations pipelines for the training split (with augmentation) and the
validation/test splits (resize and normalization only).


In [12]:
# ================== ELA (Error Level Analysis) ==================
def compute_ela(image_bgr, quality=90):
    """Compute Error Level Analysis map.

    Re-saves image as JPEG at given quality, then computes absolute
    difference to reveal JPEG compression inconsistencies between
    authentic and tampered regions.

    Args:
        image_bgr: Input image in BGR format (as loaded by cv2.imread).
        quality: JPEG re-save quality (default 90).

    Returns:
        Grayscale ELA map as uint8 numpy array.
    """
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encoded = cv2.imencode('.jpg', image_bgr, encode_param)
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    ela = cv2.absdiff(image_bgr, decoded)
    return cv2.cvtColor(ela, cv2.COLOR_BGR2GRAY)

In [13]:
# ================== Define transforms ==================
IMAGE_SIZE = CONFIG['image_size']

def get_train_transform():
    """Augmentation pipeline for training images, masks, and ELA maps."""
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.25),
        A.ImageCompression(quality_range=(50, 90), p=0.25),
        A.Affine(
            translate_percent={'x': (-0.02, 0.02), 'y': (-0.02, 0.02)},
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5,
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'ela': 'image'})

def get_valid_transform():
    """Deterministic preprocessing for validation and test samples."""
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'ela': 'image'})

### 6.2 Dataset Class

The `ELAImageMaskDataset` class loads image-mask pairs from metadata, computes ELA maps,
applies shared transforms, and returns 4-channel tensors (RGB + ELA) compatible with the
TamperDetector model.

In [14]:
# ================== Dataset with ELA channel ==================
class ELAImageMaskDataset(Dataset):
    """Dataset that loads RGB images, computes ELA, and returns 4-channel input."""

    def __init__(self, df, transform=None, ela_quality=90):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.ela_quality = ela_quality

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        mask_path = row['mask_path']
        label = int(row['label'])

        # Load image in BGR (OpenCV default)
        image_bgr = cv2.imread(img_path)
        if image_bgr is None:
            raise RuntimeError(f'Failed to read image: {img_path}')

        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.zeros((image_bgr.shape[0], image_bgr.shape[1]), dtype=np.uint8)

        # Compute ELA before color conversion
        ela = compute_ela(image_bgr, quality=self.ela_quality)

        # Convert to RGB for augmentation
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        mask = (mask > 0).astype('float32')

        # Convert ELA to 3-channel for Albumentations compatibility
        ela_3ch = np.stack([ela, ela, ela], axis=-1)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask, ela=ela_3ch)
            image_t = augmented['image']       # (3, H, W) normalized
            mask_t  = augmented['mask']         # (H, W)
            ela_t   = augmented['ela']          # (3, H, W) — take first channel
        else:
            image_t = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask_t  = torch.from_numpy(mask).float()
            ela_t   = torch.from_numpy(ela_3ch).permute(2, 0, 1).float() / 255.0

        # Extract single ELA channel and normalize to [0, 1]
        ela_ch = ela_t[0:1]  # (1, H, W) — already normalized by Albumentations

        # Stack RGB + ELA = 4 channels
        input_tensor = torch.cat([image_t, ela_ch], dim=0)  # (4, H, W)

        if mask_t.ndim == 2:
            mask_t = mask_t.unsqueeze(0)

        return input_tensor, mask_t, torch.tensor(label, dtype=torch.long)

### 6.3 DataLoader Construction

DataLoaders with persistent workers, seeded worker init, and drop_last for training stability.

In [15]:
def seed_worker(worker_id):
    """Ensure each DataLoader worker uses a unique but reproducible seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(CONFIG['seed'])

train_dataset = ELAImageMaskDataset(train_df, transform=get_train_transform(),
                                     ela_quality=CONFIG['ela_quality'])
val_dataset   = ELAImageMaskDataset(val_df,   transform=get_valid_transform(),
                                     ela_quality=CONFIG['ela_quality'])
test_dataset  = ELAImageMaskDataset(test_df,  transform=get_valid_transform(),
                                     ela_quality=CONFIG['ela_quality'])

common_loader_args = dict(
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g,
    persistent_workers=True,
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, drop_last=True, **common_loader_args)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                          shuffle=False, **common_loader_args)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'],
                          shuffle=False, **common_loader_args)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}')

Train batches: 275 | Val batches: 60 | Test batches: 60
Train samples: 8829 | Val samples: 1892 | Test samples: 1893


### 6.4 Data Visualization

Visual sanity check before training: sample images with their masks and augmentations,
class distribution across splits, and image size statistics.

In [16]:
# ================== Pre-training data visualization ==================
import matplotlib.pyplot as plt
import numpy as np

def _denorm(img_t):
    """Reverse ImageNet normalization for display (handles 4-ch RGB+ELA)."""
    if img_t.shape[0] == 4:
        img_t = img_t[:3]  # extract RGB, drop ELA channel
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img = img_t.permute(1, 2, 0).numpy() * std + mean
    return np.clip(img, 0, 1)

# --- 1) Sample grid: 4 authentic + 4 tampered with masks ---
fig, axes = plt.subplots(4, 3, figsize=(10, 13))
fig.suptitle('Sample Images (top 2 authentic, bottom 2 tampered)', fontsize=13)

auth_indices = [i for i in range(len(train_dataset)) if train_df.iloc[i]['label'] == 0][:2]
tamp_indices = [i for i in range(len(train_dataset)) if train_df.iloc[i]['label'] == 1][:2]

for row, idx in enumerate(auth_indices + tamp_indices):
    img, mask, label = train_dataset[idx]
    lbl_str = 'Authentic' if label == 0 else 'Tampered'
    axes[row, 0].imshow(_denorm(img))
    axes[row, 0].set_title(f'{lbl_str} (idx {idx})')
    axes[row, 1].imshow(mask.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth Mask')
    axes[row, 2].imshow(_denorm(img))
    axes[row, 2].imshow(mask.squeeze().numpy(), cmap='Reds', alpha=0.4)
    axes[row, 2].set_title('Overlay')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
plt.show()

# --- 2) Class distribution per split ---
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar(['Authentic', 'Tampered'], [counts.get(0, 0), counts.get(1, 0)],
                  color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'{name} (n={len(df)})')
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', fontsize=9)
fig.suptitle('Class Distribution Across Splits', fontsize=13)
plt.tight_layout()
plt.show()

# --- 3) Augmentation preview: same image with 4 random augmentations ---
aug_idx = tamp_indices[0]  # pick first tampered sample
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Augmentation Preview (same tampered image, 4 random transforms)', fontsize=12)
raw_row = train_df.iloc[aug_idx]
raw_img = cv2.cvtColor(cv2.imread(raw_row['image_path']), cv2.COLOR_BGR2RGB)
raw_img_resized = cv2.resize(raw_img, (IMAGE_SIZE, IMAGE_SIZE))
axes[0].imshow(raw_img_resized)
axes[0].set_title('Original')
axes[0].axis('off')
aug_tf = get_train_transform()
raw_mask = cv2.imread(raw_row['mask_path'], cv2.IMREAD_GRAYSCALE) if pd.notna(raw_row.get('mask_path')) else np.zeros((raw_img.shape[0], raw_img.shape[1]), dtype=np.uint8)
for j in range(1, 5):
    ela_raw = compute_ela(cv2.cvtColor(raw_img, cv2.COLOR_RGB2BGR), quality=CONFIG['ela_quality'])
    ela_3ch = np.stack([ela_raw, ela_raw, ela_raw], axis=-1)
    augmented = aug_tf(image=raw_img, mask=raw_mask, ela=ela_3ch)
    aug_img = augmented['image']
    axes[j].imshow(_denorm(aug_img))
    axes[j].set_title(f'Aug {j}')
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## 7. Model Architecture

A dual-head model (`TamperDetector`) combining an SMP UNet with a pretrained ResNet34 encoder
for segmentation and a custom classification head on the bottleneck features.

```
              Input (4 x 256 x 256)
              [RGB + ELA channel]
                      |
            +-------------------+
            |  ResNet34 Encoder  | (ImageNet pretrained)
            |                   |
            |  Stage 0: 64ch    | -> skip_0
            |  Stage 1: 64ch    | -> skip_1
            |  Stage 2: 128ch   | -> skip_2
            |  Stage 3: 256ch   | -> skip_3
            |  Stage 4: 512ch   | -> bottleneck
            +--+-------------+--+
               |             |
    +----------+             +-----------+
    | DECODER                   CLASSIFIER |
    |                                      |
    | Up(512->256) + skip_3      AdaptiveAvgPool(1x1)
    | Up(256->128) + skip_2           Flatten
    | Up(128->64) + skip_1       Linear(512->256)
    | Up(64->32) + skip_0         ReLU + Dropout
    |                          Linear(256->2)
    | Conv(32->1, 1x1)               |
    |       |                        v
    v       v              cls_logits (B x 2)
seg_logits (B x 1 x 256 x 256)
```

**Key design choices:**
- **Pretrained ResNet34 encoder** — ImageNet features provide rich low-level edge/texture detectors
  that forensic detection builds on. v6.5 proved this achieves Tam-F1=0.41 vs 0.22 from scratch.
- **4-channel input (RGB + ELA)** — SMP auto-adapts the first conv layer for >3 channels.
- **Dual heads** — classification head on bottleneck (AUC=0.91 in vK.10.6), segmentation via decoder.
- **~24.5M parameters** — smaller than the custom UNet (31.6M) yet stronger due to pretrained features.

In [17]:
# ================== TamperDetector: SMP UNet + Classification Head ==================
class TamperDetector(nn.Module):
    """
    Dual-head model for tampered image detection and localization.

    Segmentation branch: SMP UNet with pretrained ResNet34 encoder.
    Classification branch: FC head on encoder bottleneck features.

    Args:
        config: Dictionary containing model hyperparameters.
    """
    def __init__(self, config):
        super().__init__()
        self.segmentor = smp.Unet(
            encoder_name=config['encoder_name'],
            encoder_weights=config['encoder_weights'],
            in_channels=config['in_channels'],
            classes=config['n_classes'],
        )

        # Classification head on bottleneck (512 for ResNet34)
        encoder_out = self.segmentor.encoder.out_channels[-1]
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(encoder_out, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(config['dropout']),
            nn.Linear(256, config['n_labels']),
        )

    def forward(self, x):
        features = self.segmentor.encoder(x)
        cls_logits = self.classifier(features[-1])
        decoder_output = self.segmentor.decoder(features)
        seg_logits = self.segmentor.segmentation_head(decoder_output)
        return cls_logits, seg_logits

In [18]:
model = TamperDetector(CONFIG).to(device)

# Wrap with DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel enabled across {torch.cuda.device_count()} GPUs')


def get_base_model(m):
    """Unwrap DataParallel to access the underlying model."""
    return m.module if isinstance(m, nn.DataParallel) else m


def freeze_encoder(m):
    """Freeze encoder parameters to protect pretrained BatchNorm statistics."""
    base = get_base_model(m)
    for param in base.segmentor.encoder.parameters():
        param.requires_grad = False


def unfreeze_encoder(m):
    """Unfreeze encoder parameters for fine-tuning."""
    base = get_base_model(m)
    for param in base.segmentor.encoder.parameters():
        param.requires_grad = True


# Verify model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Shape verification
with torch.no_grad():
    dummy = torch.randn(1, CONFIG['in_channels'], CONFIG['image_size'], CONFIG['image_size']).to(device)
    cls_out, seg_out = model(dummy)
    assert seg_out.shape == (1, 1, CONFIG['image_size'], CONFIG['image_size']), \
        f'Unexpected seg shape: {seg_out.shape}'
    assert cls_out.shape == (1, CONFIG['n_labels']), \
        f'Unexpected cls shape: {cls_out.shape}'
print(f'Shape check passed: ({CONFIG["in_channels"]}, {CONFIG["image_size"]}, {CONFIG["image_size"]}) -> seg {seg_out.shape}, cls {cls_out.shape}')

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

DataParallel enabled across 2 GPUs
Total parameters:     24,571,347
Trainable parameters: 24,571,347
Shape check passed: (4, 256, 256) -> seg torch.Size([1, 1, 256, 256]), cls torch.Size([1, 2])


## 8. Experiment Tracking

W&B is attached for experiment tracking.  Controlled by `CONFIG['use_wandb']`.
Falls back to offline mode if Kaggle secrets are unavailable.


In [19]:
import importlib.util
import subprocess

WANDB_ACTIVE = False
WANDB_RUN = None

if CONFIG['use_wandb']:
    WANDB_CONFIG = {k: v for k, v in CONFIG.items()}

    try:
        if importlib.util.find_spec("wandb") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])

        import wandb

        try:
            from kaggle_secrets import UserSecretsClient
            wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
            if not wandb_api_key:
                raise ValueError('WANDB_API_KEY is empty')
            wandb.login(key=wandb_api_key)
            WANDB_RUN = wandb.init(
                project="vK.11.1-tampered-image-detection-assignment",
                name=f"vK.11.1-smp-resnet34-ela-seed{SEED}-kaggle",
                tags=["vk11.1", "smp", "resnet34", "ela", "edge-loss", "amp", "early-stopping", "multi-gpu", "eval-suite"],
                config=WANDB_CONFIG,
                reinit=True,
            )
            WANDB_ACTIVE = True
        except Exception as auth_exc:
            print(f"W&B online unavailable, switching to offline: {auth_exc}")
            WANDB_RUN = wandb.init(
                project="tampered-image-detection-assignment",
                name="vk11.1-offline",
                config=WANDB_CONFIG,
                mode="offline",
                reinit=True,
            )
            WANDB_ACTIVE = True
    except Exception as exc:
        print(f"W&B setup failed: {exc}")

print(f"W&B active: {WANDB_ACTIVE}")

if WANDB_ACTIVE:
    wandb.config.update({
        'notebook_version': 'vK.11.1',
        'device': str(device),
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'num_params': sum(p.numel() for p in get_base_model(model).parameters()),
    }, allow_val_change=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: harsh_vardhan (harsh_vardhan_iiitn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run t6czno53
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260314_140441-t6czno53
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run vK.11.1-smp-resnet34-ela-seed42-kaggle
wandb: ⭐️ View project at https://wandb.ai/harsh_vardhan_iii

W&B active: True


## 9. Training Utilities

This section defines loss functions (Focal, BCE+Dice, Edge), evaluation metrics,
the optimizer (AdamW with differential LR), scheduler (ReduceLROnPlateau),
checkpoint helpers, and the AMP scaler.

In [20]:
# ================== Loss functions, optimizer, scheduler, AMP scaler ==================

# Compute class weights from the training split
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)


class FocalLoss(nn.Module):
    """Focal-style classification loss that down-weights easy examples."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def dice_loss(pred, target, eps=1e-7):
    """Per-sample Dice loss for segmentation logits (from v8 improvement)."""
    prob = torch.sigmoid(pred)
    # Per-sample computation to avoid batch-level bias toward large masks
    inter = (prob * target).sum(dim=(2, 3))
    union = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2.0 * inter + eps) / (union + eps)
    return (1 - dice).mean()


def edge_loss(pred_logits, gt_masks):
    """Sobel-based edge supervision loss (from Docs v11 I7).

    Computes BCE between predicted and ground truth mask edges to improve
    boundary delineation around tampered regions.
    """
    pred_prob = torch.sigmoid(pred_logits)
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                           dtype=torch.float32, device=pred_prob.device).view(1, 1, 3, 3)
    sobel_y = sobel_x.transpose(2, 3)
    # Ground truth edges
    gt_edge = (F.conv2d(gt_masks, sobel_x, padding=1).abs() +
               F.conv2d(gt_masks, sobel_y, padding=1).abs()).clamp(0, 1)
    # Predicted edges
    pred_edge = (F.conv2d(pred_prob, sobel_x, padding=1).abs() +
                 F.conv2d(pred_prob, sobel_y, padding=1).abs()).clamp(0, 1)
    with torch.amp.autocast('cuda', enabled=False):
        return F.binary_cross_entropy(pred_edge.float(), gt_edge.float())


criterion_cls = FocalLoss(alpha=class_weights, gamma=CONFIG['focal_gamma'])
bce_loss      = nn.BCEWithLogitsLoss()
ALPHA = CONFIG['alpha']
BETA  = CONFIG['beta']
GAMMA = CONFIG['gamma']

# AdamW with differential learning rates (encoder=1e-4, decoder/heads=1e-3)
base_model = get_base_model(model)
optimizer = torch.optim.AdamW([
    {'params': base_model.segmentor.encoder.parameters(), 'lr': CONFIG['encoder_lr']},
    {'params': base_model.segmentor.decoder.parameters(), 'lr': CONFIG['decoder_lr']},
    {'params': base_model.segmentor.segmentation_head.parameters(), 'lr': CONFIG['decoder_lr']},
    {'params': base_model.classifier.parameters(), 'lr': CONFIG['decoder_lr']},
], weight_decay=CONFIG['weight_decay'])

# ReduceLROnPlateau — monitors val tampered F1 (replaces CosineAnnealing double-cycle bug)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max',
    patience=CONFIG['scheduler_patience'],
    factor=CONFIG['scheduler_factor'],
)

# AMP scaler
scaler = GradScaler('cuda', enabled=CONFIG['use_amp'])

print(f'Optimizer: AdamW(encoder_lr={CONFIG["encoder_lr"]}, decoder_lr={CONFIG["decoder_lr"]}, wd={CONFIG["weight_decay"]})')
print(f'Scheduler: ReduceLROnPlateau(patience={CONFIG["scheduler_patience"]}, factor={CONFIG["scheduler_factor"]})')
print(f'AMP: {"enabled" if CONFIG["use_amp"] else "disabled"}')
print(f'Loss weights: alpha={ALPHA}, beta={BETA}, gamma={GAMMA}')
print(f'Gradient accumulation: {CONFIG["accumulation_steps"]} steps')

Class weights: tensor([0.8420, 1.2310], device='cuda:0')
Optimizer: AdamW(encoder_lr=0.0001, decoder_lr=0.001, wd=0.0001)
Scheduler: ReduceLROnPlateau(patience=3, factor=0.5)
AMP: enabled
Loss weights: alpha=1.5, beta=1.0, gamma=0.3
Gradient accumulation: 4 steps


### 9.1 Evaluation Metrics

Dice, IoU, and F1 are computed on thresholded binary masks.  To address metric
inflation from authentic images (where both prediction and ground truth are empty),
`compute_metrics_split()` reports metrics **separately for tampered-only samples**.


In [21]:
# ================== Evaluation Metrics ==================

def dice_coef(pred, target, eps=1e-7):
    """Dice coefficient for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return dice.mean().item()


def iou_coef(pred, target, eps=1e-7):
    """IoU for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()


def f1_coef(pred, target, eps=1e-7):
    """Pixel-level F1 for thresholded segmentation predictions."""
    prob = torch.sigmoid(pred)
    pred_bin = (prob > CONFIG['seg_threshold']).float()
    tp = (pred_bin * target).sum(dim=(1,2,3))
    fp = (pred_bin * (1.0 - target)).sum(dim=(1,2,3))
    fn = ((1.0 - pred_bin) * target).sum(dim=(1,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return (2.0 * precision * recall / (precision + recall + eps)).mean().item()


def compute_metrics_split(seg_logits, masks, labels):
    """Compute metrics for all samples and tampered-only samples separately."""
    all_dice, all_iou, all_f1 = [], [], []
    tam_dice, tam_iou, tam_f1 = [], [], []

    for i in range(seg_logits.size(0)):
        sl = seg_logits[i:i+1]
        m  = masks[i:i+1]
        d  = dice_coef(sl, m)
        io = iou_coef(sl, m)
        f  = f1_coef(sl, m)
        all_dice.append(d)
        all_iou.append(io)
        all_f1.append(f)

        if labels[i].item() == 1:  # tampered only
            tam_dice.append(d)
            tam_iou.append(io)
            tam_f1.append(f)

    return {
        'dice': np.mean(all_dice),
        'iou': np.mean(all_iou),
        'f1': np.mean(all_f1),
        'tampered_dice': np.mean(tam_dice) if tam_dice else 0.0,
        'tampered_iou': np.mean(tam_iou) if tam_iou else 0.0,
        'tampered_f1': np.mean(tam_f1) if tam_f1 else 0.0,
    }

### 9.2 Checkpoint Helpers

Three-file checkpoint strategy:
- `last_checkpoint.pt` — saved every epoch for seamless resume
- `best_model.pt` — saved when tampered-only Dice improves
- `checkpoint_epoch_N.pt` — periodic snapshot every N epochs


In [22]:
def save_checkpoint(state, filepath):
    """Save training state to a checkpoint file."""
    torch.save(state, filepath)


def load_checkpoint(filepath, model, optimizer, scaler, scheduler=None):
    """Restore training state from a checkpoint file."""
    ckpt = torch.load(filepath, map_location=device, weights_only=False)
    get_base_model(model).load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    restored_history = ckpt.get('history', None)
    return ckpt['epoch'] + 1, ckpt.get('best_metric', 0.0), ckpt.get('best_epoch', 0), restored_history


print('Checkpoint helpers defined.')

Checkpoint helpers defined.


## 10. Training Loop

The training loop uses:
- **AMP** for mixed precision training
- **Gradient accumulation** for larger effective batch sizes
- **Gradient clipping** at `max_grad_norm` for stability
- **Encoder freeze** for the first N epochs to protect pretrained BN statistics
- **ReduceLROnPlateau** scheduler monitoring tampered-only F1
- **Three-file checkpointing** with automatic resume
- **Early stopping** based on tampered-only Dice coefficient

In [23]:
def train_one_epoch(epoch):
    """Train for one epoch with AMP, gradient accumulation, and edge loss."""
    model.train()
    running_loss = 0.0
    correct = 0
    total_samples = 0
    accum_steps = CONFIG['accumulation_steps']

    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, masks, labels) in enumerate(tqdm(train_loader, desc=f'Train Ep{epoch}', leave=False)):
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = CONFIG['seg_bce_weight'] * bce_loss(seg_logits, masks) + \
                       CONFIG['seg_dice_weight'] * dice_loss(seg_logits, masks)
            loss_edge = edge_loss(seg_logits, masks)
            loss = (ALPHA * loss_cls + BETA * loss_seg + GAMMA * loss_edge) / accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * accum_steps * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()
        total_samples += images.size(0)

    # Flush partial accumulation window
    if (batch_idx + 1) % accum_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['max_grad_norm'])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct / total_samples

    return epoch_loss, epoch_acc

In [24]:
@torch.no_grad()
def evaluate(loader, dataset_len, name='Val'):
    """Evaluate model with AMP, returning all-sample and tampered-only metrics."""
    model.eval()
    running_loss = 0.0
    correct = 0
    all_cls_logits, all_seg_logits, all_masks, all_labels = [], [], [], []

    for images, masks, labels in tqdm(loader, desc=name, leave=False):
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = CONFIG['seg_bce_weight'] * bce_loss(seg_logits, masks) + \
                       CONFIG['seg_dice_weight'] * dice_loss(seg_logits, masks)
            loss_edge = edge_loss(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg + GAMMA * loss_edge

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()

        all_cls_logits.append(cls_logits.cpu())
        all_seg_logits.append(seg_logits.cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())

    all_cls_logits = torch.cat(all_cls_logits)
    all_seg_logits = torch.cat(all_seg_logits)
    all_masks = torch.cat(all_masks)
    all_labels = torch.cat(all_labels)

    seg_metrics = compute_metrics_split(all_seg_logits, all_masks, all_labels)

    epoch_loss = running_loss / dataset_len
    epoch_acc = correct / dataset_len

    # ROC-AUC for classification
    probs = torch.softmax(all_cls_logits, dim=1)[:, 1].numpy()
    labels_np = all_labels.numpy()
    try:
        auc = roc_auc_score(labels_np, probs)
    except ValueError:
        auc = 0.0

    seg_metrics['loss'] = epoch_loss
    seg_metrics['acc'] = epoch_acc
    seg_metrics['roc_auc'] = auc

    print(
        f'  {name} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | AUC: {auc:.4f} | '
        f'Dice(tam): {seg_metrics["tampered_dice"]:.4f} | '
        f'IoU(tam): {seg_metrics["tampered_iou"]:.4f}'
    )
    return seg_metrics

In [25]:
# ================== Training state initialization ==================
history = {
    "train_loss": [], "train_acc": [], "train_dice": [],
    "val_loss": [], "val_acc": [],
    "val_dice": [], "val_iou": [], "val_f1": [],
    "val_tampered_dice": [], "val_tampered_iou": [], "val_tampered_f1": [],
    "val_roc_auc": [],
    "lr": [],
}

best_metric = 0.0  # tampered-only Dice
best_epoch = 0
start_epoch = 1

# Resume from checkpoint if available
resume_path = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')
if os.path.exists(resume_path):
    start_epoch, best_metric, best_epoch, restored_history = load_checkpoint(
        resume_path, model, optimizer, scaler, scheduler
    )
    if restored_history:
        history = restored_history
    print(f'Resumed from epoch {start_epoch}, best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
else:
    print('Starting fresh training.')


Starting fresh training.


In [26]:
# ================== Main training loop ==================
best_model_path = os.path.join(str(CHECKPOINT_DIR), 'best_model.pt')

# Encoder freeze for warmup
if CONFIG['encoder_freeze_epochs'] > 0:
    freeze_encoder(model)
    print(f"Encoder FROZEN for first {CONFIG['encoder_freeze_epochs']} epochs")

for epoch in range(start_epoch, CONFIG['max_epochs'] + 1):
    print(f'\nEpoch {epoch}/{CONFIG["max_epochs"]}')

    # Unfreeze encoder after warmup period
    if epoch == start_epoch + CONFIG['encoder_freeze_epochs']:
        unfreeze_encoder(model)
        print("Encoder UNFROZEN for fine-tuning")

    train_loss, train_acc = train_one_epoch(epoch)
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')

    val_metrics = evaluate(val_loader, len(val_dataset), name='Val')

    # Step scheduler with monitored metric (ReduceLROnPlateau)
    scheduler.step(val_metrics['tampered_f1'])

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_dice'].append(0.0)  # train dice computed at eval time if needed
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['acc'])
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_tampered_dice'].append(val_metrics['tampered_dice'])
    history['val_tampered_iou'].append(val_metrics['tampered_iou'])
    history['val_tampered_f1'].append(val_metrics['tampered_f1'])
    history['val_roc_auc'].append(val_metrics['roc_auc'])
    history['lr'].append(optimizer.param_groups[0]['lr'])

    # W&B logging
    if WANDB_ACTIVE:
        wandb.log({
            'epoch': epoch,
            'train/loss': train_loss, 'train/accuracy': train_acc,
            'val/loss': val_metrics['loss'], 'val/accuracy': val_metrics['acc'],
            'val/dice': val_metrics['dice'], 'val/iou': val_metrics['iou'],
            'val/f1': val_metrics['f1'],
            'val/tampered_dice': val_metrics['tampered_dice'],
            'val/tampered_iou': val_metrics['tampered_iou'],
            'val/tampered_f1': val_metrics['tampered_f1'],
            'val/roc_auc': val_metrics['roc_auc'],
            'lr/encoder': optimizer.param_groups[0]['lr'],
            'lr/decoder': optimizer.param_groups[1]['lr'],
        })
        # --- W&B: log sample predictions every 5 epochs ---
        if epoch % 5 == 0 or epoch == start_epoch:
            try:
                import matplotlib.pyplot as _plt_v
                import numpy as _np_v
                _viz_images = []
                _v_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                _v_std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                with torch.no_grad():
                    for _vb_imgs, _vb_masks, _vb_labels in val_loader:
                        for _vi in range(_vb_imgs.size(0)):
                            if _vb_labels[_vi].item() == 1 and len(_viz_images) < 2:
                                _inp = _vb_imgs[_vi:_vi+1].to(device)
                                with autocast('cuda', enabled=CONFIG['use_amp']):
                                    _, _seg_out = model(_inp)
                                _pred = torch.sigmoid(_seg_out).cpu().squeeze().numpy()
                                _rgb = (_vb_imgs[_vi, :3] * _v_std + _v_mean).clamp(0, 1).permute(1, 2, 0).numpy()
                                _gt = _vb_masks[_vi].squeeze().numpy()
                                _ov = _rgb.copy()
                                _ov[_pred > 0.5] = _ov[_pred > 0.5] * 0.5 + _np_v.array([1., 0., 0.]) * 0.5
                                _fig_v, _ax_v = _plt_v.subplots(1, 4, figsize=(16, 4))
                                _ax_v[0].imshow(_rgb); _ax_v[0].set_title('Original'); _ax_v[0].axis('off')
                                _ax_v[1].imshow(_gt, cmap='gray'); _ax_v[1].set_title('GT Mask'); _ax_v[1].axis('off')
                                _ax_v[2].imshow(_pred, cmap='hot'); _ax_v[2].set_title('Predicted'); _ax_v[2].axis('off')
                                _ax_v[3].imshow(_ov); _ax_v[3].set_title('Overlay'); _ax_v[3].axis('off')
                                _fig_v.suptitle(f'Epoch {epoch} - Sample {len(_viz_images)+1}', fontsize=12)
                                _plt_v.tight_layout()
                                _viz_images.append(wandb.Image(_fig_v))
                                _plt_v.close(_fig_v)
                        if len(_viz_images) >= 2:
                            break
                if _viz_images:
                    wandb.log({'val_predictions': _viz_images}, commit=False)
            except Exception:
                pass

    # Build checkpoint state (save unwrapped model for portability)
    state = {
        'epoch': epoch,
        'model_state_dict': get_base_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'config': CONFIG,
        'history': history,
    }

    # Save last checkpoint every epoch
    save_checkpoint(state, os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt'))

    # Save history CSV every epoch for crash resilience
    pd.DataFrame(history).to_csv(os.path.join(RESULTS_DIR, 'training_history.csv'), index=False)

    # Best model selection: tampered-only Dice
    current_metric = val_metrics['tampered_dice']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch
        state['best_metric'] = best_metric
        state['best_epoch'] = best_epoch
        save_checkpoint(state, best_model_path)
        torch.save(get_base_model(model).state_dict(),
                   os.path.join(str(KAGGLE_WORKING_DIR), 'best_model.pth'))
        print(f'  ** New best model: tampered Dice = {best_metric:.4f} at epoch {epoch}')

    # Periodic checkpoint
    if epoch % CONFIG['checkpoint_every'] == 0:
        save_checkpoint(state, os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pt'))
        print(f'  Periodic checkpoint saved at epoch {epoch}')

    # Early stopping
    if epoch - best_epoch >= CONFIG['patience']:
        print(f'Early stopping at epoch {epoch}. Best tampered Dice={best_metric:.4f} at epoch {best_epoch}')
        break

print(f'\nTraining complete. Best tampered Dice={best_metric:.4f} at epoch {best_epoch}')

Encoder FROZEN for first 2 epochs

Epoch 1/50


Train Ep1:   0%|          | 0/275 [00:00<?, ?it/s]

  Train Loss: 4.6094 | Train Acc: 0.5257


Val:   0%|          | 0/60 [00:00<?, ?it/s]

  Val Loss: 1.7452 | Acc: 0.4091 | AUC: 0.6670 | Dice(tam): 0.1246 | IoU(tam): 0.0748
  ** New best model: tampered Dice = 0.1246 at epoch 1

Epoch 2/50


...

  Val Loss: 1.8042 | Acc: 0.4434 | AUC: 0.6524 | Dice(tam): 0.1300 | IoU(tam): 0.0797

Epoch 14/50


Train Ep14:   0%|          | 0/275 [00:00<?, ?it/s]

  Train Loss: 1.6950 | Train Acc: 0.4594


Val:   0%|          | 0/60 [00:00<?, ?it/s]

  Val Loss: 1.8258 | Acc: 0.4366 | AUC: 0.6483 | Dice(tam): 0.1320 | IoU(tam): 0.0813
Early stopping at epoch 14. Best tampered Dice=0.1362 at epoch 4

Training complete. Best tampered Dice=0.1362 at epoch 4


## 11. Evaluation

Load the best checkpoint and evaluate on the held-out test split.
Reports both all-sample and tampered-only metrics.


In [27]:
# ================== Load best model and evaluate on test set ==================
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    get_base_model(model).load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model from epoch {ckpt["best_epoch"]}')
else:
    get_base_model(model).load_state_dict(torch.load(best_model_path, map_location=device))
    print('Loaded best model from legacy path')

test_metrics = evaluate(test_loader, len(test_dataset), name='Test')

print(f'\n{"="*50}')
print('FINAL TEST RESULTS')
print(f'{"="*50}')
print(f'Accuracy:         {test_metrics["acc"]:.4f}')
print(f'Dice (all):       {test_metrics["dice"]:.4f}')
print(f'IoU (all):        {test_metrics["iou"]:.4f}')
print(f'F1 (all):         {test_metrics["f1"]:.4f}')
print(f'Dice (tampered):  {test_metrics["tampered_dice"]:.4f}')
print(f'IoU (tampered):   {test_metrics["tampered_iou"]:.4f}')
print(f'F1 (tampered):    {test_metrics["tampered_f1"]:.4f}')
print(f'ROC-AUC:          {test_metrics["roc_auc"]:.4f}')

TRAINING_HISTORY = history
FINAL_TEST_METRICS = test_metrics

if WANDB_ACTIVE:
    wandb.summary.update({
        'best_epoch': best_epoch,
        'test/accuracy': test_metrics['acc'],
        'test/dice': test_metrics['dice'],
        'test/tampered_dice': test_metrics['tampered_dice'],
        'test/tampered_iou': test_metrics['tampered_iou'],
        'test/tampered_f1': test_metrics['tampered_f1'],
        'test/roc_auc': test_metrics['roc_auc'],
    })
    # --- W&B: log model artifact ---
    try:
        _ckpt = os.path.join(str(KAGGLE_WORKING_DIR), 'best_model.pth')
        if os.path.exists(_ckpt):
            _art = wandb.Artifact('best-model', type='model',
                description=f'Best checkpoint from epoch {best_epoch}')
            _art.add_file(_ckpt)
            WANDB_RUN.log_artifact(_art)
            print('  W&B: model artifact logged')
    except Exception as _e:
        print(f'  W&B artifact skip: {_e}')

    # --- W&B: save training history CSV ---
    try:
        _hist = os.path.join(str(RESULTS_DIR), 'training_history.csv')
        if os.path.exists(str(_hist)):
            wandb.save(str(_hist))
            print('  W&B: training_history.csv saved')
    except Exception as _e:
        print(f'  W&B save skip: {_e}')

    # --- W&B: log final metrics table ---
    try:
        _tbl = wandb.Table(
            columns=['Metric', 'Value'],
            data=[
                ['accuracy', test_metrics['acc']],
                ['dice_all', test_metrics['dice']],
                ['tampered_dice', test_metrics['tampered_dice']],
                ['tampered_iou', test_metrics['tampered_iou']],
                ['tampered_f1', test_metrics['tampered_f1']],
                ['roc_auc', test_metrics['roc_auc']],
                ['best_epoch', float(best_epoch)],
            ]
        )
        wandb.log({'final_test_metrics_table': _tbl})
        print('  W&B: final metrics table logged')
    except Exception as _e:
        print(f'  W&B table skip: {_e}')

Loaded best model from epoch 4


Test:   0%|          | 0/60 [00:00<?, ?it/s]

  Test Loss: 1.7504 | Acc: 0.5235 | AUC: 0.6550 | Dice(tam): 0.1274 | IoU(tam): 0.0780

FINAL TEST RESULTS
Accuracy:         0.5235
Dice (all):       0.0517
IoU (all):        0.0317
F1 (all):         0.0517
Dice (tampered):  0.1274
IoU (tampered):   0.0780
F1 (tampered):    0.1274
ROC-AUC:          0.6550


wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


  W&B: model artifact logged
  W&B: training_history.csv saved
  W&B: final metrics table logged


### 11.1 Metric Inflation Note

**Why tampered-only metrics matter:** Authentic images have all-zero ground truth masks.
A model that predicts all-zeros on authentic images gets perfect Dice/IoU for those samples,
inflating aggregate scores.  The tampered-only metrics isolate localization quality on images
that actually contain manipulated regions.


### 11.2 Training Curves

In [28]:
history_df = pd.DataFrame(history) if history['train_loss'] else pd.read_csv(
    os.path.join(RESULTS_DIR, 'training_history.csv'))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

epochs_range = history_df.index + 1

# Loss curves
axes[0,0].plot(epochs_range, history_df['train_loss'], label='Train Loss')
axes[0,0].plot(epochs_range, history_df['val_loss'], label='Val Loss')
axes[0,0].set_title('Loss Curves')
axes[0,0].set_xlabel('Epoch')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Segmentation metrics (all + tampered-only)
if 'train_dice' in history_df.columns:
    axes[0,1].plot(epochs_range, history_df['train_dice'], label='Train Dice (tam)', ls='--', alpha=0.5)
axes[0,1].plot(epochs_range, history_df['val_dice'], label='Val Dice (all)', ls='--', alpha=0.5)
axes[0,1].plot(epochs_range, history_df['val_tampered_dice'], label='Dice (tampered)', lw=2)
axes[0,1].plot(epochs_range, history_df['val_tampered_iou'], label='IoU (tampered)')
axes[0,1].plot(epochs_range, history_df['val_tampered_f1'], label='F1 (tampered)')
axes[0,1].set_title('Segmentation Metrics')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Accuracy
axes[1,0].plot(epochs_range, history_df['train_acc'], label='Train Acc')
axes[1,0].plot(epochs_range, history_df['val_acc'], label='Val Acc')
axes[1,0].set_title('Image-Level Accuracy')
axes[1,0].set_xlabel('Epoch')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Learning rate
axes[1,1].plot(epochs_range, history_df['lr'], label='LR', color='orange')
axes[1,1].set_title('Learning Rate Schedule')
axes[1,1].set_xlabel('Epoch')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

if WANDB_ACTIVE:
    wandb.log({'training_curves': wandb.Image(fig)})

### 11.3 Segmentation Threshold Optimization

Sweep the segmentation threshold from 0.05 to 0.80 on the **validation set** and select the threshold maximizing tampered-only F1. Then re-evaluate the test set with the optimal threshold.

In [29]:
# ================== Threshold Sweep on Validation Set (50-point) ==================
@torch.no_grad()
def collect_predictions(loader):
    """Collect all predictions and ground truths from a dataloader."""
    model.eval()
    all_seg_logits, all_masks, all_labels, all_cls_logits = [], [], [], []
    for images, masks, labels in tqdm(loader, desc='Collecting predictions', leave=False):
        images = images.to(device)
        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(images)
        all_seg_logits.append(seg_logits.cpu())
        all_masks.append(masks.cpu())
        all_labels.append(labels.cpu())
        all_cls_logits.append(cls_logits.cpu())
    return {
        'seg_logits': torch.cat(all_seg_logits),
        'masks': torch.cat(all_masks),
        'labels': torch.cat(all_labels),
        'cls_logits': torch.cat(all_cls_logits),
    }

val_preds = collect_predictions(val_loader)
test_preds = collect_predictions(test_loader)

thresholds = np.linspace(0.05, 0.80, 50)
val_f1_scores = []
for thr in thresholds:
    pred_bin = (torch.sigmoid(val_preds['seg_logits']) > thr).float()
    tam_mask = val_preds['labels'] == 1
    if tam_mask.sum() == 0:
        val_f1_scores.append(0.0)
        continue
    tp = (pred_bin[tam_mask] * val_preds['masks'][tam_mask]).sum(dim=(1,2,3))
    fp = (pred_bin[tam_mask] * (1 - val_preds['masks'][tam_mask])).sum(dim=(1,2,3))
    fn = ((1 - pred_bin[tam_mask]) * val_preds['masks'][tam_mask]).sum(dim=(1,2,3))
    eps = 1e-7
    prec = (tp + eps) / (tp + fp + eps)
    rec = (tp + eps) / (tp + fn + eps)
    val_f1_scores.append((2 * prec * rec / (prec + rec + eps)).mean().item())

optimal_idx = np.argmax(val_f1_scores)
OPTIMAL_THRESHOLD = float(thresholds[optimal_idx])
print(f"Optimal threshold: {OPTIMAL_THRESHOLD:.4f} (val tampered F1 = {val_f1_scores[optimal_idx]:.4f})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, val_f1_scores, 'b-o', markersize=3)
ax.axvline(OPTIMAL_THRESHOLD, color='r', linestyle='--', label=f'Optimal = {OPTIMAL_THRESHOLD:.4f}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Default = 0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('Tampered-Only F1')
ax.set_title('Segmentation Threshold Sweep (Validation Set, 50 points)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if WANDB_ACTIVE:
    wandb.log({'evaluation/threshold_sweep': wandb.Image(fig)})
    wandb.summary.update({'optimal_threshold': OPTIMAL_THRESHOLD})

def compute_metrics_at_threshold(seg_logits, masks, labels, threshold):
    eps = 1e-7
    results = {}
    for name, filt in [('all', None), ('tampered', labels == 1)]:
        sl = seg_logits[filt] if filt is not None else seg_logits
        ms = masks[filt] if filt is not None else masks
        if len(sl) == 0:
            results[f'{name}_dice'] = 0.0
            results[f'{name}_iou'] = 0.0
            results[f'{name}_f1'] = 0.0
            continue
        pred_bin = (torch.sigmoid(sl) > threshold).float()
        tp = (pred_bin * ms).sum(dim=(1,2,3))
        fp = (pred_bin * (1 - ms)).sum(dim=(1,2,3))
        fn = ((1 - pred_bin) * ms).sum(dim=(1,2,3))
        inter = tp
        union = pred_bin.sum(dim=(1,2,3)) + ms.sum(dim=(1,2,3))
        results[f'{name}_dice'] = ((2 * inter + eps) / (union + eps)).mean().item()
        results[f'{name}_iou'] = ((inter + eps) / (union - inter + eps)).mean().item()
        prec = (tp + eps) / (tp + fp + eps)
        rec = (tp + eps) / (tp + fn + eps)
        results[f'{name}_f1'] = ((2 * prec * rec) / (prec + rec + eps)).mean().item()
    return results

test_opt = compute_metrics_at_threshold(test_preds['seg_logits'], test_preds['masks'],
                                         test_preds['labels'], OPTIMAL_THRESHOLD)
print(f"\nTest Metrics at Optimal Threshold ({OPTIMAL_THRESHOLD:.4f}):")
for k, v in test_opt.items():
    print(f"  {k}: {v:.4f}")

Optimal threshold: 0.0500 (val tampered F1 = 0.1412)

Test Metrics at Optimal Threshold (0.0500):
  all_dice: 0.0537
  all_iou: 0.0335
  all_f1: 0.0537
  tampered_dice: 0.1321
  tampered_iou: 0.0825
  tampered_f1: 0.1321


### 11.4 Pixel-Level AUC-ROC

Threshold-independent localization quality metric.

In [30]:
# ================== Pixel-Level AUC-ROC ==================
from sklearn.metrics import roc_auc_score as sk_roc_auc_score

tam_mask = test_preds['labels'] == 1
tam_seg_probs = torch.sigmoid(test_preds['seg_logits'][tam_mask]).numpy().flatten()
tam_gt_masks = test_preds['masks'][tam_mask].numpy().flatten()
pixel_auc = sk_roc_auc_score(tam_gt_masks, tam_seg_probs)

cls_probs = torch.softmax(test_preds['cls_logits'], dim=1)[:, 1].numpy()
image_auc = sk_roc_auc_score(test_preds['labels'].numpy(), cls_probs)

print(f"Pixel-Level AUC-ROC (tampered only): {pixel_auc:.4f}")
print(f"Image-Level AUC-ROC (all samples):   {image_auc:.4f}")

Pixel-Level AUC-ROC (tampered only): 0.4482
Image-Level AUC-ROC (all samples):   0.6550


### 11.5 Classification Analysis: Confusion Matrix, ROC Curve, PR Curve

In [31]:
# ================== Confusion Matrix + ROC/PR Curves ==================
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, average_precision_score
import seaborn as sns

cls_probs_test = torch.softmax(test_preds['cls_logits'], dim=1)[:, 1].numpy()
cls_preds_test = torch.argmax(test_preds['cls_logits'], dim=1).numpy()
labels_test = test_preds['labels'].numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(labels_test, cls_preds_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Authentic', 'Tampered'], yticklabels=['Authentic', 'Tampered'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Image-Level)')

fpr, tpr, _ = roc_curve(labels_test, cls_probs_test)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC = {image_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Image-Level)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

precision_arr, recall_arr, _ = precision_recall_curve(labels_test, cls_probs_test)
ap = average_precision_score(labels_test, cls_probs_test)
axes[2].plot(recall_arr, precision_arr, 'g-', lw=2, label=f'AP = {ap:.4f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve (Image-Level)')
axes[2].legend(loc='lower left')
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if WANDB_ACTIVE:
    wandb.log({'evaluation/confusion_matrix_roc_pr': wandb.Image(fig)})

### 11.6 Per-Forgery-Type Evaluation

CASIA filenames: `Tp_D_*` = copy-move, `Tp_S_*` = splicing.

In [32]:
# ================== Per-Forgery-Type Evaluation ==================
thr = OPTIMAL_THRESHOLD
forgery_types = []
for path in test_df['image_path'].tolist():
    fname = os.path.basename(path)
    if fname.startswith('Tp_D'): forgery_types.append('copy-move')
    elif fname.startswith('Tp_S'): forgery_types.append('splicing')
    elif fname.startswith('Au'): forgery_types.append('authentic')
    else: forgery_types.append('unknown')
forgery_types = np.array(forgery_types)

print("Forgery type distribution in test set:")
for ft in ['authentic', 'splicing', 'copy-move']:
    print(f"  {ft}: {(forgery_types == ft).sum()}")

print(f"\nPer-type metrics (threshold={thr:.2f}):")
print(f"{'Type':<15} {'Count':>6} {'Dice':>8} {'IoU':>8} {'F1':>8}")
print('-' * 50)
seg_probs = torch.sigmoid(test_preds['seg_logits'])
for ft in ['splicing', 'copy-move']:
    ft_m = forgery_types == ft
    if ft_m.sum() == 0: continue
    pb = (seg_probs[ft_m] > thr).float()
    gt = test_preds['masks'][ft_m]
    eps = 1e-7
    inter = (pb*gt).sum(dim=(1,2,3))
    dice = ((2*inter+eps)/(pb.sum(dim=(1,2,3))+gt.sum(dim=(1,2,3))+eps)).mean().item()
    iou = ((inter+eps)/(pb.sum(dim=(1,2,3))+gt.sum(dim=(1,2,3))-inter+eps)).mean().item()
    tp = inter; fp = (pb*(1-gt)).sum(dim=(1,2,3)); fn = ((1-pb)*gt).sum(dim=(1,2,3))
    pr = (tp+eps)/(tp+fp+eps); rc = (tp+eps)/(tp+fn+eps)
    f1 = (2*pr*rc/(pr+rc+eps)).mean().item()
    print(f"  {ft:<13} {ft_m.sum():>6} {dice:>8.4f} {iou:>8.4f} {f1:>8.4f}")

Forgery type distribution in test set:
  authentic: 1124
  splicing: 509
  copy-move: 260

Per-type metrics (threshold=0.05):
Type             Count     Dice      IoU       F1
--------------------------------------------------
  splicing         509   0.1016   0.0614   0.1016
  copy-move        260   0.1918   0.1239   0.1918


### 11.7 Mask-Size Stratified Evaluation

Bucket by mask area: tiny (<2%), small (2-5%), medium (5-15%), large (>15%).

In [33]:
# ================== Mask-Size Stratified Evaluation ==================
thr = OPTIMAL_THRESHOLD
tam_indices = (test_preds['labels'] == 1).nonzero(as_tuple=True)[0]
mask_areas = np.array([test_preds['masks'][i].sum().item()/test_preds['masks'][i].numel()*100 for i in tam_indices])

buckets = [('tiny (<2%)',0,2),('small (2-5%)',2,5),('medium (5-15%)',5,15),('large (>15%)',15,100)]
print(f"Mask-size stratified evaluation (threshold={thr:.2f}):")
print(f"{'Bucket':<18} {'Count':>6} {'Dice':>8} {'F1':>8}")
print('-' * 44)
bucket_names, bucket_f1s, bucket_counts = [], [], []
for bname, lo, hi in buckets:
    sel = (mask_areas >= lo) & (mask_areas < hi)
    if sel.sum() == 0:
        print(f"  {bname:<16} {0:>6}      -        -")
        continue
    idx = tam_indices[sel]
    pb = (torch.sigmoid(test_preds['seg_logits'][idx]) > thr).float()
    gt = test_preds['masks'][idx]
    eps = 1e-7
    inter = (pb*gt).sum(dim=(1,2,3))
    dice = ((2*inter+eps)/(pb.sum(dim=(1,2,3))+gt.sum(dim=(1,2,3))+eps)).mean().item()
    tp = inter; fp = (pb*(1-gt)).sum(dim=(1,2,3)); fn = ((1-pb)*gt).sum(dim=(1,2,3))
    pr = (tp+eps)/(tp+fp+eps); rc = (tp+eps)/(tp+fn+eps)
    f1 = (2*pr*rc/(pr+rc+eps)).mean().item()
    print(f"  {bname:<16} {sel.sum():>6} {dice:>8.4f} {f1:>8.4f}")
    bucket_names.append(bname); bucket_f1s.append(f1); bucket_counts.append(int(sel.sum()))

if bucket_names:
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(range(len(bucket_names)), bucket_f1s, color='steelblue')
    ax.set_xticks(range(len(bucket_names)))
    ax.set_xticklabels(bucket_names)
    ax.set_ylabel('Tampered-Only F1')
    ax.set_title('F1 by Mask Size Bucket')
    ax.set_ylim(0, 1)
    for bar, cnt in zip(bars, bucket_counts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'n={cnt}', ha='center', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    if WANDB_ACTIVE:
        wandb.log({'evaluation/mask_size_f1': wandb.Image(fig)})

Mask-size stratified evaluation (threshold=0.05):
Bucket              Count     Dice       F1
--------------------------------------------
  tiny (<2%)          289   0.0190   0.0190
  small (2-5%)        190   0.0630   0.0630
  medium (5-15%)      171   0.1537   0.1537
  large (>15%)        119   0.4859   0.4859


### 11.8 Shortcut Learning Checks

1. **Mask randomization:** Shuffle GT masks, check F1 drop.
2. **Boundary sensitivity:** Erode predictions by 1px, check Dice stability.

In [34]:
# ================== Shortcut Learning Checks ==================
thr = OPTIMAL_THRESHOLD
tam_idx = (test_preds['labels'] == 1).nonzero(as_tuple=True)[0]
tam_seg = test_preds['seg_logits'][tam_idx]
tam_masks_sc = test_preds['masks'][tam_idx]
eps = 1e-7

def compute_tam_f1(pred_bin, gt_masks):
    tp = (pred_bin * gt_masks).sum(dim=(1,2,3))
    fp = (pred_bin * (1 - gt_masks)).sum(dim=(1,2,3))
    fn = ((1 - pred_bin) * gt_masks).sum(dim=(1,2,3))
    pr = (tp+eps)/(tp+fp+eps); rc = (tp+eps)/(tp+fn+eps)
    return (2*pr*rc/(pr+rc+eps)).mean().item()

pred_bin = (torch.sigmoid(tam_seg) > thr).float()
baseline_f1 = compute_tam_f1(pred_bin, tam_masks_sc)

shuffled_masks = tam_masks_sc[torch.randperm(len(tam_masks_sc))]
shuffled_f1 = compute_tam_f1(pred_bin, shuffled_masks)

eroded_pred = -F.max_pool2d(-pred_bin, kernel_size=3, stride=1, padding=1)
eroded_f1 = compute_tam_f1(eroded_pred, tam_masks_sc)

print("SHORTCUT LEARNING VALIDATION")
print("=" * 55)
print(f"  Baseline tampered F1:       {baseline_f1:.4f}")
print(f"  Shuffled-mask F1:           {shuffled_f1:.4f}  (delta: {shuffled_f1-baseline_f1:+.4f})")
print(f"  Eroded-prediction F1:       {eroded_f1:.4f}  (delta: {eroded_f1-baseline_f1:+.4f})")
print()
if baseline_f1 - shuffled_f1 > 0.05:
    print("  [PASS] Mask randomization: F1 drops -> model uses image content")
else:
    print("  [WARN] Mask randomization: F1 stable -> possible shortcut learning")
if abs(baseline_f1 - eroded_f1) < 0.10:
    print("  [PASS] Boundary sensitivity: F1 stable under erosion")
else:
    print("  [INFO] Boundary sensitivity: F1 changes with erosion")

SHORTCUT LEARNING VALIDATION
  Baseline tampered F1:       0.1321
  Shuffled-mask F1:           0.1321  (delta: +0.0000)
  Eroded-prediction F1:       0.1321  (delta: +0.0000)

  [WARN] Mask randomization: F1 stable -> possible shortcut learning
  [PASS] Boundary sensitivity: F1 stable under erosion


## 12. Visualization of Predictions

The following cells collect authentic and tampered examples from the test loader and visualize model outputs.
For assignment submission, the notebook makes the qualitative evidence clearer by surfacing:

- original image
- ground-truth mask
- predicted mask
- overlay highlighting predicted manipulated regions

Together, these views show both detection and localization performance.

**Assignment alignment:** This section provides qualitative evidence that the model detects tampering and highlights the manipulated region.


In [35]:
def denormalize(img_tensor):
    """Convert a normalized image tensor back to displayable RGB space.

    Handles 4-channel (RGB+ELA) tensors by extracting only the RGB channels.
    """
    if img_tensor.shape[0] == 4:
        img_tensor = img_tensor[:3]  # extract RGB channels only
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

In [36]:
# Load best model for visualization
if os.path.exists(best_model_path):
    ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
    get_base_model(model).load_state_dict(ckpt['model_state_dict'])
    print(f'Best model loaded from checkpoint (epoch {ckpt.get("best_epoch", "?")})')
else:
    legacy_path = os.path.join(str(KAGGLE_WORKING_DIR), 'best_model.pth')
    get_base_model(model).load_state_dict(torch.load(legacy_path, map_location=device))
    print('Best model loaded from legacy path')
model.eval()

Best model loaded from checkpoint (epoch 4)


DataParallel(
  (module): TamperDetector(
    (segmentor): Unet(
      (encoder): ResNetEncoder(
        (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (layer1): Sequential(
          (0): BasicBlock(
            (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (relu): ReLU(inplace=True)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (1): BasicBlock(
            (conv1): Conv2d(64, 64, kernel_size=(3, 3)

### 12.1 Sample Collection for Qualitative Review

The next helper functions collect balanced authentic and tampered examples from the test split and convert predictions into reviewer-friendly visualizations.
This makes it easier to compare image-level decisions with the corresponding predicted tampering regions.

**Assignment alignment:** This subsection connects the evaluation outputs to qualitative evidence for both detection and localization.


In [37]:
def collect_samples(loader, num_real=5, num_fake=5):
    """
    Purpose:
        Gather a balanced set of authentic and tampered examples for qualitative visualization.

    Inputs:
        loader (DataLoader): Dataloader that provides images, masks, and image-level labels.
        num_real (int): Number of authentic examples to collect.
        num_fake (int): Number of tampered examples to collect.

    Returns:
        tuple: Two lists containing authentic samples and tampered samples.

    Notes:
        Each sample stores the input image, ground-truth mask, predicted mask probabilities, and image-level labels.
    """
    real_samples = []
    fake_samples = []

    with torch.no_grad():
        for images, masks, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass: compute both image-level and pixel-level predictions for qualitative review.
            cls_logits, seg_logits = model(images)
            seg_probs = torch.sigmoid(seg_logits)
            preds = torch.argmax(cls_logits, dim=1)

            for i in range(images.size(0)):
                sample = {
                    "image": images[i].cpu(),
                    "mask_true": masks[i].cpu(),
                    "mask_pred": seg_probs[i].cpu(),
                    "label": int(labels[i].item()),
                    "pred": int(preds[i].item())
                }

                if sample["label"] == 0 and len(real_samples) < num_real:
                    real_samples.append(sample)

                if sample["label"] == 1 and len(fake_samples) < num_fake:
                    fake_samples.append(sample)

                if len(real_samples) >= num_real and len(fake_samples) >= num_fake:
                    return real_samples, fake_samples

    return real_samples, fake_samples


real_samples, fake_samples = collect_samples(test_loader, 5, 5)

print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

Collected Real: 5  Fake: 5


In [38]:
import matplotlib.pyplot as plt
import numpy as np

def show_samples_with_masks(samples, title):
    """
    Purpose:
        Visualize predicted tampered regions as red overlays on top of the original image.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the panel.

    Returns:
        None.

    Notes:
        Overlay views help the reader see where the model believes manipulation evidence is concentrated.
    """
    n = len(samples)
    plt.figure(figsize=(4*n, 4))

    for i, s in enumerate(samples):
        img = denormalize(s["image"]).permute(1,2,0).numpy()
        # Threshold the predicted probability map to produce a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Color predicted tampered pixels in red so the localization output is easy to interpret.
        overlay[mask_pred==1] = [1, 0, 0]

        blended = (0.6 * img + 0.4 * overlay)

        plt.subplot(1, n, i+1)
        plt.imshow(blended)
        lbl = "Real" if s["label"]==0 else "Fake"
        pred_lbl = "Real" if s["pred"]==0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_samples_with_masks(real_samples, "5 Real Images with Predicted Manipulation Regions")
show_samples_with_masks(fake_samples, "5 Fake Images with Predicted Manipulation Regions")

In [39]:
import matplotlib.pyplot as plt
import numpy as np

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display each image together with its predicted binary tampering mask.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample is shown in two rows: the original image first, then the thresholded predicted mask.
    """
    n = len(samples)
    plt.figure(figsize=(6*n, 6))

    for idx, s in enumerate(samples):
        # Convert the normalized tensor back to a displayable RGB image.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to obtain a black-and-white mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Top row: original image with image-level ground truth and prediction labels.
        plt.subplot(2, n, idx+1)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

        # Bottom row: predicted binary mask for tampered-region localization.
        plt.subplot(2, n, n + idx + 1)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Predicted Mask")
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [40]:
show_image_and_mask(real_samples, "5 Real Images + Predicted Masks")
show_image_and_mask(fake_samples, "5 Fake Images + Predicted Masks")


### 12.2 ELA Channel Visualization

Visualize the Error Level Analysis maps alongside RGB images and predictions
to demonstrate the forensic signal the model receives as its 4th input channel.

In [41]:
# ================== ELA Visualization ==================
def show_ela_visualization(loader, num_samples=3):
    """Show RGB, ELA, predicted mask, and overlay for sample images."""
    model.eval()
    fig, axes = plt.subplots(num_samples * 2, 4, figsize=(16, 4 * num_samples * 2))
    shown_auth, shown_tamp = 0, 0
    row = 0

    with torch.no_grad():
        for images, masks, labels in loader:
            for i in range(images.size(0)):
                if row >= num_samples * 2:
                    break
                label = labels[i].item()
                if label == 0 and shown_auth >= num_samples:
                    continue
                if label == 1 and shown_tamp >= num_samples:
                    continue

                # Extract RGB (first 3 channels) and ELA (4th channel)
                img_rgb = images[i, :3].clone()
                ela_ch = images[i, 3].clone()

                # Denormalize RGB for display
                mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                img_display = (img_rgb * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

                # ELA channel for display
                ela_display = ela_ch.numpy()

                # Prediction
                inp = images[i:i+1].to(device)
                with autocast('cuda', enabled=CONFIG['use_amp']):
                    _, seg_logits = model(inp)
                pred = torch.sigmoid(seg_logits).cpu().squeeze().numpy()

                # GT mask
                gt = masks[i].squeeze().numpy()

                tag = 'Authentic' if label == 0 else 'Tampered'
                axes[row, 0].imshow(img_display)
                axes[row, 0].set_title(f'{tag} - RGB')
                axes[row, 0].axis('off')

                axes[row, 1].imshow(ela_display, cmap='hot')
                axes[row, 1].set_title(f'{tag} - ELA')
                axes[row, 1].axis('off')

                axes[row, 2].imshow(gt, cmap='gray')
                axes[row, 2].set_title('Ground Truth')
                axes[row, 2].axis('off')

                axes[row, 3].imshow(pred, cmap='hot')
                axes[row, 3].set_title('Prediction')
                axes[row, 3].axis('off')

                row += 1
                if label == 0:
                    shown_auth += 1
                else:
                    shown_tamp += 1

            if row >= num_samples * 2:
                break

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'ela_visualization.png'), dpi=150, bbox_inches='tight')
    plt.show()

    if WANDB_ACTIVE:
        wandb.log({'evaluation/ela_visualization': wandb.Image(fig)})

show_ela_visualization(test_loader, num_samples=3)

In [42]:
real_samples, fake_samples = collect_samples(test_loader, num_real=10, num_fake=10)
print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

import matplotlib.pyplot as plt
import numpy as np
import math

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display image and predicted-mask pairs in a grid that limits each row to three samples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample occupies two neighboring columns: one for the image and one for the predicted mask.
    """
    total = len(samples)
    cols = 3
    rows = math.ceil(total / cols)

    plt.figure(figsize=(cols * 6, rows * 4))

    for idx, s in enumerate(samples):
        row = idx // cols
        col = idx % cols

        # Convert the normalized tensor back to RGB for display.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to form a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Compute the first subplot column used by this sample inside the current row.
        base_col = col * 2

        img_pos  = row * cols * 2 + base_col + 1
        mask_pos = img_pos + 1

        # Show the original image and the image-level prediction.
        plt.subplot(rows, cols*2, img_pos)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}", fontsize=10)
        plt.axis("off")

        # Show the corresponding thresholded localization mask.
        plt.subplot(rows, cols*2, mask_pos)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Mask", fontsize=10)
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

show_image_and_mask(real_samples, "10 Real Images (3 per row)")
show_image_and_mask(fake_samples, "10 Fake Images (3 per row)")

Collected Real: 10  Fake: 10


### 12.2 Submission-Ready Prediction Panels

The final visualization block packages the qualitative outputs into a compact four-panel format.
Each row aligns the original image, ground-truth mask, predicted mask, and overlay so the assignment reviewer can inspect localization quality quickly.

**Assignment alignment:** This subsection supports the final qualitative presentation requirement of the assignment.


In [43]:
import matplotlib.pyplot as plt
import numpy as np

def show_submission_prediction_grid(samples, title, max_items=6):
    """
    Purpose:
        Create a submission-style four-panel view for a small set of qualitative examples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.
        max_items (int): Maximum number of examples to visualize.

    Returns:
        matplotlib.figure.Figure | None: The generated figure when samples are available.

    Notes:
        Each row shows the original image, ground-truth mask, predicted mask, and overlay for side-by-side review.
    """
    chosen = samples[:max_items]
    if not chosen:
        print(f"No samples available for: {title}")
        return None

    fig, axes = plt.subplots(len(chosen), 4, figsize=(16, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    column_titles = ["Original Image", "Ground Truth Mask", "Predicted Mask", "Overlay"]

    for row_idx, sample in enumerate(chosen):
        img = denormalize(sample["image"]).permute(1, 2, 0).numpy()
        gt_mask = (sample["mask_true"][0].numpy() > 0.5).astype(np.uint8)
        pred_mask = (sample["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Highlight predicted tampered pixels in red to make the localization decision easy to inspect.
        overlay[pred_mask == 1] = [1, 0, 0]
        blended = 0.6 * img + 0.4 * overlay

        panels = [img, gt_mask, pred_mask, blended]
        cmaps = [None, "gray", "gray", None]

        for col_idx, panel in enumerate(panels):
            ax = axes[row_idx, col_idx]
            if cmaps[col_idx] is None:
                ax.imshow(panel)
            else:
                ax.imshow(panel, cmap=cmaps[col_idx])
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(column_titles[col_idx])

        lbl = "Authentic" if sample["label"] == 0 else "Tampered"
        pred_lbl = "Authentic" if sample["pred"] == 0 else "Tampered"
        axes[row_idx, 0].set_ylabel(f"GT: {lbl}\nPred: {pred_lbl}", fontsize=10)

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()
    return fig

real_fig = show_submission_prediction_grid(real_samples, "Submission Grid: Authentic Image Predictions", max_items=4)
fake_fig = show_submission_prediction_grid(fake_samples, "Submission Grid: Tampered Image Predictions", max_items=4)

if WANDB_ACTIVE:
    if real_fig is not None:
        wandb.log({"submission_grid_authentic": wandb.Image(real_fig)})
    if fake_fig is not None:
        wandb.log({"submission_grid_tampered": wandb.Image(fake_fig)})

## 13. Advanced Analysis

### 13.1 Failure Case Analysis

Display the 10 worst predictions (lowest per-sample Dice) among tampered test images.

In [44]:
# ================== Failure Case Analysis ==================
thr = OPTIMAL_THRESHOLD
tam_idx_fc = (test_preds['labels'] == 1).nonzero(as_tuple=True)[0]

per_sample_dice = []
for i in tam_idx_fc:
    sl = test_preds['seg_logits'][i:i+1]
    m = test_preds['masks'][i:i+1]
    pb = (torch.sigmoid(sl) > thr).float()
    eps = 1e-7
    per_sample_dice.append(((2*(pb*m).sum()+eps)/(pb.sum()+m.sum()+eps)).item())
per_sample_dice = np.array(per_sample_dice)
worst_order = np.argsort(per_sample_dice)[:10]

fig, axes = plt.subplots(min(10, len(worst_order)), 4, figsize=(16, 4*min(10, len(worst_order))))
if len(worst_order) == 1: axes = axes[np.newaxis, :]
fig.suptitle('10 Worst Predictions (Lowest Dice)', fontsize=16, y=1.01)

for row, wi in enumerate(worst_order):
    gi = tam_idx_fc[wi].item()
    img_path = test_df.iloc[gi]['image_path']
    fname = os.path.basename(img_path)
    ftype = 'copy-move' if fname.startswith('Tp_D') else 'splicing' if fname.startswith('Tp_S') else 'unknown'
    mask_area = test_preds['masks'][gi].sum().item()/test_preds['masks'][gi].numel()*100
    orig = cv2.imread(img_path)
    if orig is not None:
        orig = cv2.cvtColor(cv2.resize(orig, (256,256)), cv2.COLOR_BGR2RGB)
    else:
        orig = np.zeros((256,256,3), dtype=np.uint8)
    gt = test_preds['masks'][gi,0].numpy()
    pred = (torch.sigmoid(test_preds['seg_logits'][gi,0]) > thr).float().numpy()
    overlay = orig.copy().astype(float)
    overlay[pred > 0.5] = overlay[pred > 0.5]*0.5 + np.array([255,0,0])*0.5
    axes[row,0].imshow(orig); axes[row,0].set_title(f'{ftype} | area={mask_area:.1f}%', fontsize=8)
    axes[row,1].imshow(gt, cmap='gray'); axes[row,1].set_title('Ground Truth', fontsize=8)
    axes[row,2].imshow(pred, cmap='gray'); axes[row,2].set_title(f'Pred (Dice={per_sample_dice[wi]:.3f})', fontsize=8)
    axes[row,3].imshow(overlay.astype(np.uint8)); axes[row,3].set_title('Overlay', fontsize=8)
    for ax in axes[row]: ax.axis('off')
plt.tight_layout()
plt.show()

if WANDB_ACTIVE:
    wandb.log({'evaluation/failure_cases': wandb.Image(fig)})

## 14. Explainability

### 14.1 Grad-CAM Visualization

Grad-CAM heatmaps from the encoder bottleneck show what the model attends to.

In [45]:
# ================== Grad-CAM Explainability (SMP Encoder) ==================
import warnings

class GradCAM:
    """Grad-CAM for segmentation encoder features."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._handles = []
        self._handles.append(target_layer.register_forward_hook(self._save_activation))
        self._handles.append(target_layer.register_full_backward_hook(self._save_gradient))

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor):
        """Generate Grad-CAM heatmap."""
        self.model.eval()
        self.gradients = None
        self.activations = None

        # Forward pass — use segmentation output for backward
        cls_out, seg_out = self.model(input_tensor)
        target = seg_out.mean()
        self.model.zero_grad()
        target.backward()

        if self.gradients is None or self.activations is None:
            warnings.warn('Grad-CAM: hooks did not capture data.')
            return None

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().numpy()

    def remove_hooks(self):
        for h in self._handles:
            h.remove()

# Target the deepest encoder layer (layer4 for ResNet34)
_base = get_base_model(model)
grad_cam = GradCAM(_base, _base.segmentor.encoder.layer4)

# Collect samples for Grad-CAM visualization
num_auth, num_tamp = 3, 3
auth_samples, tamp_samples = [], []

with torch.no_grad():
    for images, masks, labels in test_loader:
        for i in range(images.size(0)):
            label = labels[i].item()
            if label == 0 and len(auth_samples) < num_auth:
                auth_samples.append({'image': images[i], 'mask': masks[i], 'label': label})
            elif label == 1 and len(tamp_samples) < num_tamp:
                tamp_samples.append({'image': images[i], 'mask': masks[i], 'label': label})
        if len(auth_samples) >= num_auth and len(tamp_samples) >= num_tamp:
            break

all_samples = auth_samples + tamp_samples
fig, axes = plt.subplots(len(all_samples), 3, figsize=(12, 4 * len(all_samples)))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for idx, sample in enumerate(all_samples):
    inp = sample['image'].unsqueeze(0).to(device).requires_grad_(True)
    cam = grad_cam.generate(inp)

    # Display RGB (first 3 channels)
    rgb = sample['image'][:3]
    img_np = (rgb * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    tag = 'Authentic' if sample['label'] == 0 else 'Tampered'
    axes[idx, 0].imshow(img_np)
    axes[idx, 0].set_title(f'{tag}')
    axes[idx, 0].axis('off')

    if cam is not None:
        axes[idx, 1].imshow(img_np)
        axes[idx, 1].imshow(cam, cmap='jet', alpha=0.5)
        axes[idx, 1].set_title('Grad-CAM')
    else:
        axes[idx, 1].set_title('Grad-CAM (failed)')
    axes[idx, 1].axis('off')

    with torch.no_grad():
        _, seg_out = model(sample['image'].unsqueeze(0).to(device))
    pred = torch.sigmoid(seg_out).cpu().squeeze().numpy()
    axes[idx, 2].imshow(pred, cmap='hot')
    axes[idx, 2].set_title('Predicted Mask')
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

if WANDB_ACTIVE:
    wandb.log({'evaluation/gradcam': wandb.Image(fig)})

grad_cam.remove_hooks()

## 15. Robustness Testing

Test the model against 8 post-processing degradation conditions at inference time.

In [46]:
# ================== Robustness Testing Suite (Albumentations-based) ==================
NORMALIZE = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

robustness_transforms = {
    'clean': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE), NORMALIZE, ToTensorV2()],
                       additional_targets={'ela': 'image'}),
    'jpeg_qf70': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                            A.ImageCompression(quality_lower=70, quality_upper=70, p=1.0),
                            NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
    'jpeg_qf50': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                            A.ImageCompression(quality_lower=50, quality_upper=50, p=1.0),
                            NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
    'noise_s10': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
                            NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
    'noise_s25': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                            A.GaussNoise(var_limit=(100.0, 100.0), p=1.0),
                            NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
    'blur_k3': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                          A.GaussianBlur(blur_limit=(3, 3), p=1.0),
                          NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
    'blur_k5': A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                          A.GaussianBlur(blur_limit=(5, 5), p=1.0),
                          NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
    'resize_0.75': A.Compose([A.Resize(int(IMAGE_SIZE*0.75), int(IMAGE_SIZE*0.75)),
                              A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                              NORMALIZE, ToTensorV2()], additional_targets={'ela': 'image'}),
}

def compute_tam_f1(pred_bin, masks, eps=1e-7):
    """Compute per-sample tampered F1."""
    tp = (pred_bin * masks).sum(dim=(1,2,3))
    fp = (pred_bin * (1 - masks)).sum(dim=(1,2,3))
    fn = ((1 - pred_bin) * masks).sum(dim=(1,2,3))
    prec = (tp + eps) / (tp + fp + eps)
    rec = (tp + eps) / (tp + fn + eps)
    return (2 * prec * rec / (prec + rec + eps)).mean().item()

@torch.no_grad()
def run_robustness_eval(condition_name, transform, threshold):
    """Evaluate model under a specific degradation condition."""
    model.eval()
    robust_dataset = ELAImageMaskDataset(
        test_df[test_df['label'] == 1],  # tampered only
        transform=transform,
        ela_quality=CONFIG['ela_quality'],
    )
    if len(robust_dataset) == 0:
        return 0.0
    robust_loader = DataLoader(robust_dataset, batch_size=CONFIG['batch_size'],
                               shuffle=False, num_workers=CONFIG['num_workers'],
                               pin_memory=True)
    all_preds, all_masks = [], []
    for images, masks, labels in robust_loader:
        images = images.to(device)
        with autocast('cuda', enabled=CONFIG['use_amp']):
            _, seg_logits = model(images)
        pred_bin = (torch.sigmoid(seg_logits).cpu() > threshold).float()
        all_preds.append(pred_bin)
        all_masks.append(masks)
    return compute_tam_f1(torch.cat(all_preds), torch.cat(all_masks))

threshold = OPTIMAL_THRESHOLD if 'OPTIMAL_THRESHOLD' in dir() else CONFIG['seg_threshold']
print(f'Robustness evaluation using threshold={threshold:.4f}')

robustness_results = {}
for name, transform in tqdm(robustness_transforms.items(), desc='Robustness tests'):
    f1 = run_robustness_eval(name, transform, threshold)
    robustness_results[name] = f1
    print(f'  {name:20s}: F1={f1:.4f}')

# Delta from clean
clean_f1 = robustness_results.get('clean', 0.0)
print(f'\nDeltas from clean (F1={clean_f1:.4f}):')
for name, f1 in robustness_results.items():
    if name != 'clean':
        print(f'  {name:20s}: delta={f1 - clean_f1:+.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names = list(robustness_results.keys())
f1s = [robustness_results[n] for n in names]
colors = ['green' if n == 'clean' else 'steelblue' for n in names]
ax.bar(names, f1s, color=colors)
ax.axhline(clean_f1, color='green', linestyle='--', alpha=0.5, label=f'Clean baseline = {clean_f1:.4f}')
ax.set_ylabel('Tampered-Only F1')
ax.set_title('Robustness Testing: F1 Under Degradation')
ax.legend()
ax.set_xticklabels(names, rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'robustness_results.png'), dpi=150, bbox_inches='tight')
plt.show()

if WANDB_ACTIVE:
    wandb.log({'evaluation/robustness': wandb.Image(fig)})
    try:
        _rob_table = wandb.Table(
            columns=['Condition', 'F1', 'Delta_from_Clean'],
            data=[[n, f, f - clean_f1] for n, f in robustness_results.items()]
        )
        wandb.log({'evaluation/robustness_table': _rob_table})
    except Exception:
        pass

Robustness evaluation using threshold=0.0500


/tmp/ipykernel_24/1030818982.py:8: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=70, quality_upper=70, p=1.0),
/tmp/ipykernel_24/1030818982.py:11: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=50, quality_upper=50, p=1.0),
/tmp/ipykernel_24/1030818982.py:14: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
/tmp/ipykernel_24/1030818982.py:17: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(100.0, 100.0), p=1.0),


Robustness tests:   0%|          | 0/8 [00:00<?, ?it/s]

  clean               : F1=0.1321
  jpeg_qf70           : F1=0.1321
  jpeg_qf50           : F1=0.1321
  noise_s10           : F1=0.1321
  noise_s25           : F1=0.1321
  blur_k3             : F1=0.1321
  blur_k5             : F1=0.1321
  resize_0.75         : F1=0.1322

Deltas from clean (F1=0.1321):
  jpeg_qf70           : delta=+0.0000
  jpeg_qf50           : delta=+0.0000
  noise_s10           : delta=+0.0000
  noise_s25           : delta=+0.0000
  blur_k3             : delta=+0.0000
  blur_k5             : delta=+0.0000
  resize_0.75         : delta=+0.0001


/tmp/ipykernel_24/1030818982.py:89: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(names, rotation=30, ha='right')


## 16. Inference Examples

A self-contained inference function for running the trained model on
individual images after training.


In [47]:
def predict_single_image(image_path, model, device, threshold=None):
    """Run inference on a single image with ELA preprocessing."""
    if threshold is None:
        threshold = OPTIMAL_THRESHOLD if 'OPTIMAL_THRESHOLD' in dir() else CONFIG['seg_threshold']

    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise RuntimeError(f'Failed to read image: {image_path}')

    # Compute ELA
    ela = compute_ela(image_bgr, quality=CONFIG['ela_quality'])
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    ela_3ch = np.stack([ela, ela, ela], axis=-1)

    # Apply validation transform
    transform = get_valid_transform()
    augmented = transform(image=image_rgb, mask=np.zeros_like(ela), ela=ela_3ch)
    image_t = augmented['image']
    ela_t = augmented['ela'][0:1]  # single channel
    input_tensor = torch.cat([image_t, ela_t], dim=0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with autocast('cuda', enabled=CONFIG['use_amp']):
            cls_logits, seg_logits = model(input_tensor)

    cls_probs = torch.softmax(cls_logits, dim=1)[0]
    seg_prob = torch.sigmoid(seg_logits).cpu().squeeze().numpy()
    seg_mask = (seg_prob > threshold).astype(np.uint8)

    return {
        'cls_probs': cls_probs.cpu().numpy(),
        'is_tampered': int(cls_probs[1] > 0.5),
        'seg_prob': seg_prob,
        'seg_mask': seg_mask,
        'threshold': threshold,
    }

print('predict_single_image() defined.')

predict_single_image() defined.


## 17. Model Card and Experiment Report

This section provides a structured project summary following the conventions
of modern ML model cards. It documents the system design, dataset,
training configuration, evaluation methodology, results, and compliance
with the Big Vision Internship Assignment requirements.

### 17.1 Model Overview

| Field | Description |
|---|---|
| **Model Name** | TamperDetector vK.11.1 |
| **Task** | Tampered image detection and pixel-level tamper localization |
| **Input** | 256 x 256 x 4 tensor (RGB + ELA channel) |
| **Outputs** | (1) Binary segmentation mask (256 x 256), (2) Image-level classification logits |
| **Architecture** | Dual-head model: SMP UNet (ResNet34 encoder, ImageNet pretrained) + FC classification head on bottleneck |
| **Framework** | PyTorch + Segmentation Models PyTorch (SMP) |
| **Approach** | Semantic segmentation with an auxiliary classification branch |

**High-level pipeline:**

1. An input image is JPEG-recompressed to produce an Error Level Analysis (ELA) map
   that highlights compression inconsistencies introduced by tampering.
2. The ELA map is stacked as a 4th channel alongside the RGB image.
3. The 4-channel input passes through a pretrained ResNet34 encoder, which extracts
   hierarchical features at multiple scales.
4. The UNet decoder fuses multi-scale encoder features to produce a pixel-level
   tamper probability map (segmentation branch).
5. The encoder bottleneck features are pooled and passed through a fully connected
   head to produce an image-level authentic/tampered classification (classification branch).
6. Both outputs are trained jointly with a combined loss:
   `L = alpha * FocalLoss(cls) + beta * (w_bce * BCE + w_dice * Dice)(seg) + gamma * EdgeLoss(seg)`

### 17.2 Dataset Summary

| Property | Value |
|---|---|
| **Dataset** | CASIA v2.0 (via Kaggle) |
| **Domain** | Image forgery detection |
| **Forgery types** | Copy-move (`Tp_D_*`) and Splicing (`Tp_S_*`) |
| **Ground truth** | Binary masks indicating tampered pixel regions |
| **Authentic images** | ~5,123 images with all-zero masks |
| **Tampered images** | ~3,706 images with non-zero masks |
| **Total images** | ~8,829 |

**Dataset split** (stratified by class label):

| Split | Proportion | Purpose |
|---|---|---|
| Train | 70% | Model training |
| Validation | 15% | Hyperparameter tuning, early stopping, threshold optimization |
| Test | 15% | Final held-out evaluation (no tuning decisions based on this split) |

Data leakage is verified at the path level: zero overlap exists between
train, validation, and test image paths (see Section 4.5).

### 17.3 Training Configuration

| Parameter | Value |
|---|---|
| **Framework** | PyTorch 2.x |
| **GPU** | Kaggle T4 (15 GB VRAM) / 2x T4 with DataParallel |
| **Input size** | 256 x 256 |
| **Batch size** | 8 (auto-adjusted by VRAM; effective = batch x accumulation_steps) |
| **Gradient accumulation** | 4 steps (effective batch = 32) |
| **Optimizer** | AdamW |
| **Encoder LR** | 1e-4 (pretrained ResNet34 — lower to preserve features) |
| **Decoder LR** | 1e-3 (randomly initialized decoder and heads) |
| **Weight decay** | 1e-4 |
| **Scheduler** | ReduceLROnPlateau (patience=3, factor=0.5, monitors val tampered F1) |
| **Max epochs** | 50 |
| **Early stopping** | Patience = 10 (based on validation tampered Dice) |
| **Encoder freeze** | First 2 epochs (protects pretrained BatchNorm statistics) |
| **Mixed precision** | AMP (automatic mixed precision) enabled |
| **Gradient clipping** | max_norm = 5.0 |
| **Reproducibility** | Seed = 42 (Python, NumPy, PyTorch, CUDA) |

**Loss function:**

```
Total Loss = alpha * FocalLoss(classification)
           + beta  * (0.5 * BCEWithLogitsLoss + 0.5 * PerSampleDiceLoss)(segmentation)
           + gamma * SobelEdgeLoss(segmentation)

alpha = 1.5, beta = 1.0, gamma = 0.3
```

**Data augmentation (training only):**

| Augmentation | Probability |
|---|---|
| HorizontalFlip | 0.5 |
| VerticalFlip | 0.3 |
| RandomRotate90 | 0.5 |
| ShiftScaleRotate | 0.3 |
| RandomBrightnessContrast | 0.3 |
| Normalize (ImageNet stats) | 1.0 |

### 17.4 Evaluation Metrics

The following metrics are used to evaluate model performance at both the
pixel level (localization) and image level (detection).

**Pixel-level segmentation metrics** (computed on thresholded binary masks):

| Metric | Definition | Why it matters |
|---|---|---|
| **Dice coefficient** | 2 * \|P ∩ G\| / (\|P\| + \|G\|) | Measures overlap between predicted and ground truth masks; penalizes both false positives and false negatives equally |
| **IoU (Jaccard)** | \|P ∩ G\| / (\|P ∪ G\|) | Stricter than Dice; standard benchmark metric for segmentation tasks |
| **Pixel F1** | 2 * Precision * Recall / (Precision + Recall) | Balances pixel-level precision and recall; equivalent to Dice for binary masks |
| **Pixel AUC-ROC** | Area under ROC at pixel level | Threshold-independent measure of localization quality |

**Image-level classification metrics:**

| Metric | Definition | Why it matters |
|---|---|---|
| **Accuracy** | Correct predictions / Total images | Overall detection rate |
| **ROC-AUC** | Area under receiver operating characteristic | Threshold-independent discriminative ability |
| **PR-AUC** | Area under precision-recall curve | Detection quality under class imbalance |

**Metric inflation warning:** Mixed-set averages (computed over all images including
authentic ones) are inflated because authentic images with all-zero masks score 1.0
when the model correctly predicts all-zero. The **tampered-only** metrics isolate
actual localization performance and are the primary evaluation criterion.

In [48]:
# ================== 17.5 Quantitative Results Summary ==================
# This cell generates the results table from the computed test metrics.
# FINAL_TEST_METRICS is populated by the evaluation cell in Section 11.

print('=' * 60)
print('         QUANTITATIVE RESULTS — MODEL CARD')
print('=' * 60)
print()

if 'FINAL_TEST_METRICS' in dir():
    m = FINAL_TEST_METRICS
    rows = [
        ('Image-Level Accuracy',           m.get('acc', 0.0)),
        ('Image-Level ROC-AUC',            m.get('roc_auc', 0.0)),
        ('Dice (all samples)',              m.get('dice', 0.0)),
        ('IoU  (all samples)',              m.get('iou', 0.0)),
        ('F1   (all samples)',              m.get('f1', 0.0)),
        ('Dice (tampered only)',            m.get('tampered_dice', 0.0)),
        ('IoU  (tampered only)',            m.get('tampered_iou', 0.0)),
        ('F1   (tampered only)',            m.get('tampered_f1', 0.0)),
    ]
    if 'OPTIMAL_THRESHOLD' in dir():
        rows.append(('Optimal Seg Threshold', OPTIMAL_THRESHOLD))
    if 'pixel_auc' in dir():
        rows.append(('Pixel-Level AUC-ROC', pixel_auc))

    print(f"{'Metric':<35} {'Score':>10}")
    print('-' * 47)
    for name, val in rows:
        print(f'{name:<35} {val:>10.4f}')
    print()
    print('Note: Tampered-only metrics are the primary evaluation criterion.')
    print('All-sample metrics are inflated by authentic images scoring 1.0.')
else:
    print('FINAL_TEST_METRICS not found — run the evaluation cells in Section 11 first.')

         QUANTITATIVE RESULTS — MODEL CARD

Metric                                   Score
-----------------------------------------------
Image-Level Accuracy                    0.5235
Image-Level ROC-AUC                     0.6550
Dice (all samples)                      0.0517
IoU  (all samples)                      0.0317
F1   (all samples)                      0.0517
Dice (tampered only)                    0.1274
IoU  (tampered only)                    0.0780
F1   (tampered only)                    0.1274
Optimal Seg Threshold                   0.0500
Pixel-Level AUC-ROC                     0.4482

Note: Tampered-only metrics are the primary evaluation criterion.
All-sample metrics are inflated by authentic images scoring 1.0.


### 17.6 Qualitative Results

The notebook includes multiple visualization outputs that provide qualitative
evidence of model performance.

**Prediction panels (Section 12):** For both authentic and tampered test images,
four-panel visualizations compare:

1. **Original image** — the input RGB image
2. **Ground truth mask** — the binary annotation of tampered regions
3. **Predicted mask** — the model's thresholded segmentation output
4. **Overlay visualization** — the predicted mask superimposed on the original image
   with color-coded true positives (green), false positives (red), and false negatives (blue)

**ELA visualization (Section 12.2):** Side-by-side display of RGB images and their
Error Level Analysis maps, showing how the ELA channel highlights compression
artifacts around tampered regions that are invisible in the RGB domain.

**Grad-CAM heatmaps (Section 14):** Gradient-weighted class activation maps from
the encoder bottleneck (`encoder.layer4`) reveal which spatial regions drove the
model's classification decision. These help verify that the model attends to
tampered regions rather than dataset artifacts.

**Training curves (Section 11.2):** Loss, accuracy, Dice, and learning rate
histories across epochs provide insight into convergence behavior.

### 17.7 Failure Analysis

Section 13.1 displays the 10 worst-performing tampered test images (lowest
per-sample Dice score). Common failure patterns observed in CASIA-2:

| Failure Mode | Description | Why it is challenging |
|---|---|---|
| **Very small tampered regions** | Forgeries that occupy less than 2% of the image area | The segmentation head must localize tiny patches against a large background; the class imbalance within each image is extreme |
| **Low contrast manipulations** | Spliced regions with similar color and texture to the surrounding area | Pixel-level differences are minimal; the model must rely on subtle compression or noise artifacts rather than visual discontinuities |
| **Complex textured backgrounds** | Tampered regions placed on grass, foliage, fabric, or other high-frequency textures | Edge-based and texture-based cues overlap with natural image content, producing false positives |
| **High-quality copy-move** | Regions duplicated from within the same image | Source and pasted regions share identical capture conditions (noise, lighting, compression), minimizing forensic traces |
| **JPEG double compression** | Whole-image re-saving obscures localized artifacts | ELA signal is suppressed when the entire image undergoes uniform re-compression |

The mask-size stratification analysis in Section 11.7 quantifies these effects:
tiny masks (<2% area) consistently achieve lower Dice scores than larger manipulations,
confirming that spatial extent is a dominant factor in localization difficulty.

The robustness evaluation in Section 15 further shows performance degradation under
controlled perturbations such as Gaussian blur, JPEG re-compression, and additive noise.

### 17.8 Assignment Requirement Compliance

The following checklist confirms compliance with the Big Vision Internship
Assignment requirements.

**Core Requirements:**

| # | Requirement | Status | Evidence |
|---|---|---|---|
| 1 | Publicly available dataset with authentic and tampered images | Complete | CASIA v2.0 from Kaggle (Section 4) |
| 2 | Dataset preparation and preprocessing | Complete | Metadata caching, ELA computation, Albumentations pipeline (Section 6) |
| 3 | Train / Validation / Test split | Complete | 70/15/15 stratified split with path-level leakage verification (Section 4.5) |
| 4 | Train a model for tampered region localization | Complete | TamperDetector with SMP UNet + pretrained ResNet34 (Section 7) |
| 5 | Architecture choice justified | Complete | Pretrained encoder proven across v6.5 (Tam-F1=0.41); dual-head design documented (Section 7) |
| 6 | T4 GPU compatible | Complete | VRAM auto-scaling, AMP, batch size adjustment (Section 2) |
| 7 | Evaluation metrics (localization + detection) | Complete | Dice, IoU, F1, Accuracy, ROC-AUC — both all-sample and tampered-only (Section 11) |
| 8 | Visual results | Complete | 4-panel prediction grids, ELA visualization, Grad-CAM heatmaps (Sections 12, 14) |
| 9 | Single notebook deliverable | Complete | All code in one self-contained notebook |
| 10 | Dataset explanation | Complete | Documented in Sections 4 and 17.2 |
| 11 | Model architecture description | Complete | Documented in Sections 7 and 17.1 |
| 12 | Training strategy documentation | Complete | CONFIG dict + summary table + this Model Card (Sections 2, 9, 17.3) |
| 13 | Clear visualizations | Complete | Training curves, prediction panels, confusion matrix, ROC/PR curves (Sections 11, 12) |

**Bonus Requirements:**

| # | Requirement | Status | Evidence |
|---|---|---|---|
| B1 | Robustness testing (JPEG, noise, blur) | Complete | 8-condition degradation suite (Section 15) |
| B2 | Subtle tampering analysis | Complete | Forgery-type breakdown + mask-size stratification (Sections 11.6, 11.7) |
| B3 | Explainability | Complete | Grad-CAM heatmaps on encoder bottleneck (Section 14) |
| B4 | Shortcut learning validation | Complete | Mask randomization + boundary erosion checks (Section 11.8) |
| B5 | Threshold optimization | Complete | 50-point sweep on validation set (Section 11.3) |
| B6 | Data leakage verification | Complete | Path-level overlap assertions (Section 4.5) |

### 17.9 Future Improvements

The following directions could further improve detection and localization quality.

**Architecture improvements:**

| Improvement | Expected Impact | Rationale |
|---|---|---|
| Transformer-based encoder (e.g., MIT-B2, Swin-T) | Higher feature quality | Self-attention captures long-range dependencies that CNNs miss; critical for detecting spatially distributed manipulations |
| Forensic residual extraction (SRM filters) | Better low-level artifact capture | Steganalysis-inspired high-pass filters amplify subtle manipulation traces in the noise domain |
| Multi-scale input / Feature Pyramid | Better small-region detection | Processing at 256 and 512 resolution simultaneously captures both fine and coarse manipulation cues |
| Test-Time Augmentation (TTA) | 2-3% F1 boost | Averaging predictions across horizontal flips and rotations reduces variance at no training cost |

**Training improvements:**

| Improvement | Expected Impact | Rationale |
|---|---|---|
| Multi-dataset training (CASIA + Columbia + Coverage) | Improved generalization | Cross-dataset diversity reduces overfitting to CASIA-2 compression artifacts |
| Hard example mining | Better tail performance | Re-weighting loss toward difficult samples (tiny masks, low-contrast splicing) addresses the long tail of failure cases |
| Larger input resolution (384 x 384) | Finer localization | v6.5 demonstrated this at the cost of smaller batch size; gradient accumulation mitigates the tradeoff |
| Knowledge distillation | Faster inference | A lighter student encoder (ResNet18, EfficientNet-B0) can approximate the ResNet34 teacher at lower latency |

**Evaluation improvements:**

| Improvement | Expected Impact | Rationale |
|---|---|---|
| Cross-dataset evaluation | Generalization measurement | Testing on unseen datasets (Columbia, Coverage, NIST16) reveals whether the model learns forensic principles or dataset biases |
| Confidence calibration | Trustworthy predictions | Calibrating predicted probabilities enables meaningful uncertainty thresholds for deployment |
| Multi-annotator agreement | Ground truth quality | CASIA-2 masks have known quality issues; measuring inter-annotator consistency bounds achievable performance |

## 18. Conclusion

This notebook (vK.11.1) represents the synthesis of findings from 13 experiment runs
across two architecture tracks:

**Architecture upgrades from audits:**
- Pretrained ResNet34 encoder via SMP (from v6.5 — proven Tam-F1=0.41)
- ELA as 4th input channel for JPEG forgery detection
- Edge supervision loss for improved boundary delineation
- AdamW with differential learning rates (from v6.5)
- ReduceLROnPlateau scheduler (from v8, fixing CosineAnnealing double-cycle bug)
- Gradient accumulation for larger effective batch sizes
- Encoder freeze warmup to protect pretrained BatchNorm statistics
- Per-sample Dice loss (from v8, fixing batch-level bias)

**Evaluation suite (12 features from vK.10.6):**
Threshold optimization, pixel-level AUC, confusion matrix, ROC/PR curves,
forgery-type breakdown, mask-size stratification, robustness testing,
Grad-CAM explainability, shortcut learning validation, failure case analysis,
data leakage verification, ELA visualization.

**Bug fixes from audit:**
- Removed stderr suppression (Bug #2)
- Fixed CONFIG/docs mismatch (Bug #1)
- Fixed unused CONFIG values (Bug #3, #4)
- Replaced CosineAnnealing double-cycle with ReduceLROnPlateau

In [49]:
if WANDB_ACTIVE and WANDB_RUN is not None:
    WANDB_RUN.finish()
    print('W&B run finished.')
else:
    print('No active W&B run to finish.')

wandb: uploading artifact run-t6czno53-evaluationrobustness_table-WUkVrg; uploading media/table/evaluation/robustness_table_25_a422288e4531cfd39825.table.json; updating run metadata
wandb: uploading artifact run-t6czno53-evaluationrobustness_table-WUkVrg; uploading wandb-summary.json
wandb: uploading artifact run-t6czno53-evaluationrobustness_table-WUkVrg; uploading wandb-summary.json; uploading config.yaml
wandb: uploading artifact run-t6czno53-evaluationrobustness_table-WUkVrg
wandb: 
wandb: Run history:
wandb:          epoch ▁▂▂▃▃▄▄▅▅▆▆▇▇█
wandb:     lr/decoder ███████▃▃▃▃▁▁▁
wandb:     lr/encoder ███████▃▃▃▃▁▁▁
wandb: train/accuracy █▄█▇▅▅▅▅▃▁▃▂▁▂
wandb:     train/loss █▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   val/accuracy ▁█▅▆█▁▂▃▁▃▁▂▂▂
wandb:       val/dice ▁▁▂█▇█▇▆▆▅▇▆▄▆
wandb:         val/f1 ▁▁▂█▇█▇▆▆▅▇▆▄▆
wandb:        val/iou ▁▁▂█▇█▇▆▆▆▇▆▅▆
wandb:       val/loss ▄▁▅▄▃▂▂▂▂▂▄█▆▇
wandb:             +4 ...
wandb: 
wandb: Run summary:
wandb:         best_epoch 4
wandb:              epoch 14
wandb: 

W&B run finished.
